# 04 · Training Deep Networks: Optimization, Initialization, Normalization, Regularization

**Part II — Training deep networks** · Estimated time: ~150 min

In Notebooks 02 and 03 we built networks and computed their gradients. Having a gradient, however, is only the beginning: turning a stream of noisy mini-batch gradients into a well-trained deep network is a craft with its own theory, its own algorithms and its own failure modes. This notebook is about that craft. We implement every optimizer that matters, derive where their update rules come from, study how the learning rate should change over time, discover why a deep network with the wrong initialization cannot learn at all, build Batch Normalization and dropout from scratch, and finish with a practical guide to diagnosing a training run and searching hyperparameters.

## Learning objectives

By the end of this notebook you will be able to:

- Explain why mini-batch gradients are noisy estimates of the true gradient, and how batch size and learning rate control that noise.
- Implement SGD, momentum, Nesterov momentum, AdaGrad, RMSProp, Adam and AdamW from scratch, derive Adam's bias correction, and explain why decoupled weight decay differs from an L2 penalty.
- Compare optimizers on two-dimensional test functions and on a real classification task, and choose sensible default hyperparameters for each.
- Implement and compare learning-rate schedules (step, exponential, cosine, warmup, one-cycle) and run a learning-rate range test.
- Derive the Xavier/Glorot and He/Kaiming initializations from variance propagation and demonstrate vanishing and exploding signals in deep networks.
- Implement Batch Normalization and inverted dropout, including their different behaviour in training and evaluation mode.
- Diagnose underfitting, overfitting, bad learning rates and dead units from loss curves and simple sanity checks.
- Set up a small random hyperparameter search with log-uniform sampling.

## Prerequisites

- Notebook 01 · Mathematical Foundations (gradients, Taylor expansion, gradient descent, conditioning, expectation and variance).
- Notebook 02 · Neural Networks from Scratch (NumPy) (multi-layer perceptrons, backpropagation, softmax cross-entropy).
- Notebook 03 · Tensors and Autograd (tensors, `requires_grad`, `backward()`, `torch.no_grad()`, a first look at `nn.Module` and `torch.optim`).

## Contents

1. The loss landscape and stochastic gradient descent
2. Optimizers from scratch and compared
3. Learning-rate schedules
4. Initialization
5. Normalization
6. Regularization
7. Diagnosing training
8. Hyperparameter search basics

Exercises · Summary · Further reading

In [ ]:
import copy
import math, os, random, time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} | device: {device}"
      + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""))

# FAST_RUN = True shrinks every training budget so the whole notebook runs in a few
# minutes on a CPU. Set it to False for the full experiments (recommended on a GPU).
FAST_RUN = False
DATA_DIR = Path("../data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

## 1. The loss landscape and stochastic gradient descent

### 1.1 What we are minimizing

Training a network means choosing parameters $\theta$ (all weights and biases, flattened into one long vector) that make a **loss** small on the training data. For a dataset of $N$ examples $(x_i, y_i)$ and a per-example loss $\ell$ (for us: softmax cross-entropy), the quantity we minimize is the **empirical risk**

$$
L(\theta) \;=\; \frac{1}{N}\sum_{i=1}^{N} \ell\big(f_\theta(x_i),\, y_i\big).
$$

In Notebook 01 · Mathematical Foundations we saw that gradient descent, $\theta \leftarrow \theta - \eta \nabla L(\theta)$, decreases $L$ for a small enough **learning rate** $\eta$ because the first-order Taylor expansion gives $L(\theta - \eta g) \approx L(\theta) - \eta \|g\|^2$. The function $L$ is the **loss landscape**: a surface over a parameter space with millions of dimensions. For neural networks it is non-convex, full of saddle points, flat plateaus and long narrow valleys. We cannot draw it, but we can reason about it locally through its gradient and curvature, and we can study two-dimensional caricatures of it (Section 2.7).

### 1.2 Full-batch versus mini-batch gradients

Computing the exact gradient $\nabla L(\theta)$ requires a forward and backward pass over all $N$ examples. For $N = 60{,}000$ images that is wasteful: most examples carry redundant information about the direction of descent. **Stochastic gradient descent (SGD)** instead samples a **mini-batch** $B \subset \{1,\dots,N\}$ of size $|B| = b$ and uses

$$
\hat g_B(\theta) \;=\; \frac{1}{b}\sum_{i \in B} \nabla_\theta\, \ell\big(f_\theta(x_i), y_i\big).
$$

If the indices are sampled uniformly, this estimate is **unbiased**, $\mathbb{E}[\hat g_B] = \nabla L(\theta)$, and its variance shrinks like $1/b$ (the per-example gradients are averaged, and the variance of a mean of $b$ roughly independent quantities is the single-example variance divided by $b$). So each step costs $b$ examples rather than $N$, at the price of **gradient noise**. Three consequences follow, and they organize much of this notebook:

1. Noise scales with $\eta / b$. A larger learning rate amplifies the random component of every step; a larger batch reduces it. Late in training, when we want to settle into a minimum, we must reduce $\eta$ (Section 3) or increase $b$.
2. A small batch performs many more updates per **epoch** (one pass over the data). For a fixed number of epochs, small batches usually reach a lower loss; for a fixed wall-clock time on a GPU, larger batches are more efficient because per-step overhead is amortized.
3. Noise is not only a nuisance. It helps escape saddle points and sharp minima, and there is good evidence that moderate noise improves generalization.

> **Key idea.** The mini-batch gradient is a noisy but unbiased estimate of the true gradient; the batch size and learning rate together set the size of the noise, and almost every trick in this notebook is a way to make progress despite (or thanks to) that noise.

### 1.3 A common experimental harness

Every method in this notebook is tested on the same task so that results are comparable: a small multi-layer perceptron classifying **FashionMNIST**, a drop-in replacement for MNIST with 28×28 grey-scale images of ten kinds of clothing. It is harder than MNIST (a linear model gets about 84%, a good MLP about 89–90%, a CNN about 93%), which leaves room for optimization choices to matter.

We download it once with `torchvision` into `DATA_DIR`, standardize the pixels, and keep the whole subset in memory as tensors on our device. Doing our own mini-batching from in-memory tensors keeps the harness fast and transparent; the `Dataset`/`DataLoader` machinery, which you want for real projects, is the subject of Notebook 05 · The PyTorch Workflow and Best Practices. Under `FAST_RUN` we use a 4,000-example training subset and a 2,000-example validation set; the full run uses 20,000 and 5,000.

In [ ]:
train_ds = datasets.FashionMNIST(root=str(DATA_DIR), train=True, download=True)
test_ds = datasets.FashionMNIST(root=str(DATA_DIR), train=False, download=True)
CLASSES = train_ds.classes
print("raw training tensor:", tuple(train_ds.data.shape), train_ds.data.dtype)

N_TRAIN = 4_000 if FAST_RUN else 20_000
N_VAL = 2_000 if FAST_RUN else 5_000

X_all = train_ds.data.float().div(255.0).view(-1, 784)      # (60000, 784), pixels in [0, 1]
y_all = train_ds.targets
perm = torch.randperm(len(X_all), generator=torch.Generator().manual_seed(SEED))
train_idx, val_idx = perm[:N_TRAIN], perm[N_TRAIN:N_TRAIN + N_VAL]

MEAN, STD = X_all[train_idx].mean(), X_all[train_idx].std()   # statistics of the training subset only
def standardize(x):
    return (x - MEAN) / STD

X_train, y_train = standardize(X_all[train_idx]).to(device), y_all[train_idx].to(device)
X_val, y_val = standardize(X_all[val_idx]).to(device), y_all[val_idx].to(device)
X_test = standardize(test_ds.data.float().div(255.0).view(-1, 784)).to(device)
y_test = test_ds.targets.to(device)
print(f"train {tuple(X_train.shape)} | val {tuple(X_val.shape)} | test {tuple(X_test.shape)}")
print(f"pixel mean {MEAN:.4f}, std {STD:.4f}; classes: {CLASSES}")

fig, axes = plt.subplots(2, 8, figsize=(10, 3))
for ax, i in zip(axes.flat, range(16)):
    ax.imshow(train_ds.data[i].cpu().numpy(), cmap="gray")
    ax.set_title(CLASSES[int(train_ds.targets[i])], fontsize=8)
    ax.axis("off")
plt.suptitle("FashionMNIST examples")
plt.tight_layout(); plt.show()

Two remarks on this cell. First, the standardization constants are computed on the **training subset only** and then applied to validation and test data; computing them on all data would leak (a tiny amount of) information from the evaluation sets into training. Second, the validation set is carved out of the official training set, and the official test set is kept untouched until the very end of the notebook, exactly as you should do in a real project.

Next we define the model factory and a training function. `make_mlp` builds a fully-connected network and always re-seeds PyTorch before creating it, so that every optimizer we compare starts from **identical initial weights** (otherwise a comparison would partly measure the luck of the draw). It already accepts `norm`, `dropout` and `init` options that we will only use from Section 4 onwards.

`train` runs mini-batch training on the in-memory tensors and returns a **history** dictionary with the loss of every step and per-epoch training/validation loss and accuracy. It accepts both our own optimizers (defined next) and `torch.optim` optimizers, because both expose the same two methods, `zero_grad()` and `step()`. The optional `scheduler`, `clip_norm`, `loss_fn` and `augment` arguments are hooks for later sections.

In [ ]:
def make_mlp(hidden=(256, 128), act=nn.ReLU, norm=None, dropout=0.0, init=None,
             in_dim=784, n_classes=10, seed=SEED):
    """Build a fresh MLP with a fixed seed so that every run starts from identical weights.

    norm: None, "batch" (BatchNorm1d) or "layer" (LayerNorm) after each hidden linear layer.
    dropout: dropout probability after each hidden activation (0 disables it).
    init: optional function applied with model.apply(init) to override PyTorch's default init.
    """
    torch.manual_seed(seed)
    layers, d = [], in_dim
    for h in hidden:
        layers.append(nn.Linear(d, h))
        if norm == "batch":
            layers.append(nn.BatchNorm1d(h))
        elif norm == "layer":
            layers.append(nn.LayerNorm(h))
        layers.append(act())
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        d = h
    layers.append(nn.Linear(d, n_classes))
    model = nn.Sequential(*layers)
    if init is not None:
        model.apply(init)
    return model.to(device)


def evaluate(model, X, y, batch_size=1000):
    """Mean cross-entropy and accuracy of `model` on (X, y): eval mode, no gradients."""
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            xb, yb = X[i:i + batch_size], y[i:i + batch_size]
            logits = model(xb)
            total_loss += F.cross_entropy(logits, yb, reduction="sum").item()
            correct += (logits.argmax(dim=1) == yb).sum().item()
    return total_loss / len(X), correct / len(X)


def current_lr(opt):
    """Learning rate of our own optimizers (attribute `lr`) or of a torch.optim optimizer."""
    return opt.lr if hasattr(opt, "lr") else opt.param_groups[0]["lr"]


def train(model, opt, epochs, X=None, y=None, batch_size=128, scheduler=None, clip_norm=None,
          loss_fn=F.cross_entropy, augment=None, verbose=True, seed=SEED):
    """Mini-batch training on in-memory tensors. Returns a history dict of per-step and per-epoch curves."""
    if X is None:
        X, y = X_train, y_train
    gen = torch.Generator().manual_seed(seed)              # controls only the shuffling
    hist = {"step_loss": [], "train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "lr": []}
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        model.train()                                      # evaluate() switches to eval mode, so switch back
        perm = torch.randperm(len(X), generator=gen).to(device)
        for i in range(0, len(X), batch_size):
            idx = perm[i:i + batch_size]
            xb, yb = X[idx], y[idx]
            if augment is not None:
                xb = augment(xb)
            loss = loss_fn(model(xb), yb)
            opt.zero_grad()
            loss.backward()
            if clip_norm is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
            opt.step()
            if scheduler is not None:
                scheduler.step()
            hist["step_loss"].append(loss.item())
        tr_loss, tr_acc = evaluate(model, X, y)
        va_loss, va_acc = evaluate(model, X_val, y_val)
        for k, v in zip(("train_loss", "train_acc", "val_loss", "val_acc", "lr"),
                        (tr_loss, tr_acc, va_loss, va_acc, current_lr(opt))):
            hist[k].append(v)
        if verbose:
            print(f"epoch {epoch:2d} | train loss {tr_loss:.4f} acc {tr_acc:.3f} | "
                  f"val loss {va_loss:.4f} acc {va_acc:.3f} | lr {current_lr(opt):.2e} | {time.time() - t0:5.1f}s")
    return hist


def smooth(x, k=20):
    """Running mean with window k (for plotting noisy per-step losses)."""
    x = np.asarray(x, dtype=float)
    if len(x) < k:
        return x
    return np.convolve(x, np.ones(k) / k, mode="valid")


EPOCHS = 3 if FAST_RUN else 10
BATCH = 128
print(f"harness: {N_TRAIN} training examples, {EPOCHS} epochs, batch size {BATCH}, "
      f"{math.ceil(N_TRAIN / BATCH)} steps per epoch")

### 1.4 SGD as a class with the `torch.optim` interface

We now write our own optimizers. A tiny base class fixes the interface: the constructor receives the parameters and a learning rate; `zero_grad()` clears the gradients; `step()` visits every parameter with a gradient and calls `update(p, g, state)`, where `state` is a per-parameter dictionary in which the subclass may keep buffers (velocities, running averages, ...). The decorator `@torch.no_grad()` is essential: parameter updates must not be recorded by autograd, otherwise the next backward pass would try to differentiate through the optimizer.

Plain SGD is then a one-liner:

$$
\theta_{t} \;=\; \theta_{t-1} - \eta\, g_t, \qquad g_t = \hat g_{B_t}(\theta_{t-1}).
$$

In [ ]:
class Optimizer:
    """Minimal base class with the same interface as torch.optim: zero_grad() and step().

    Subclasses implement `update(p, g, state)`: the in-place update of one parameter tensor
    `p` given its gradient `g`, using the per-parameter dict `state` for any buffers.
    """
    def __init__(self, params, lr):
        self.params = [p for p in params if p.requires_grad]
        self.lr = lr
        self.state = [dict() for _ in self.params]
        self.t = 0                                    # number of steps taken so far

    def zero_grad(self):
        for p in self.params:
            p.grad = None

    @torch.no_grad()
    def step(self):
        self.t += 1
        for p, s in zip(self.params, self.state):
            if p.grad is not None:
                self.update(p, p.grad, s)

    def update(self, p, g, state):
        raise NotImplementedError


class SGD(Optimizer):
    """theta <- theta - lr * g"""
    def update(self, p, g, state):
        p.sub_(self.lr * g)


model = make_mlp()
opt = SGD(model.parameters(), lr=0.1)
print(f"model: {sum(p.numel() for p in model.parameters()):,} parameters, "
      f"{len(opt.params)} parameter tensors")
hist_sgd = train(model, opt, EPOCHS)

The loss starts near $\ln 10 \approx 2.303$, the cross-entropy of a uniform guess over ten classes, and drops quickly. Even plain SGD gets a respectable accuracy on this problem within a few epochs; the interesting question is how *fast* and how *reliably* each method gets there, which is what the rest of the notebook measures.

### 1.5 Experiment: the effect of batch size

Let us make the noise argument of Section 1.2 visible. We train the same model with the same learning rate and batch sizes 16, 64 and 256, and plot the per-step loss of each run against the epoch count. Note that the number of updates per epoch differs by a factor of 16 between the smallest and the largest batch.

In [ ]:
bs_results = {}
for bs in (16, 64, 256):
    model = make_mlp()
    opt = SGD(model.parameters(), lr=0.1)
    t0 = time.time()
    bs_results[bs] = train(model, opt, EPOCHS, batch_size=bs, verbose=False)
    h = bs_results[bs]
    print(f"batch {bs:4d}: {len(h['step_loss']) // EPOCHS:4d} steps/epoch | "
          f"final val acc {h['val_acc'][-1]:.3f} | {time.time() - t0:.1f}s")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
for bs, h in bs_results.items():
    steps = np.arange(len(h["step_loss"]))
    ax1.plot(steps / (len(steps) / EPOCHS), h["step_loss"], lw=0.6, alpha=0.8, label=f"batch {bs}")
    ax2.plot(range(1, EPOCHS + 1), h["val_acc"], "o-", label=f"batch {bs}")
ax1.set_xlabel("epoch"); ax1.set_ylabel("mini-batch loss"); ax1.set_title("Per-step loss (raw)")
ax1.set_ylim(0, 2.5); ax1.legend()
ax2.set_xlabel("epoch"); ax2.set_ylabel("validation accuracy"); ax2.set_title("Validation accuracy per epoch")
ax2.legend()
plt.tight_layout(); plt.show()

The left panel shows the noise directly: the batch-16 curve is a thick band because each mini-batch loss is a high-variance estimate, while the batch-256 curve is a thin line. The right panel shows the other side of the trade-off: at a fixed number of epochs the small batch has taken sixteen times more steps and is typically ahead in validation accuracy, although each of its steps is cheaper and noisier. On a GPU the per-epoch wall-clock time favours large batches, because a batch of 256 costs barely more than a batch of 16 per step. Practitioners therefore raise the batch size to fill the hardware and compensate with a larger learning rate: the **linear scaling rule** (Goyal et al., 2017) multiplies $\eta$ by $k$ when the batch is multiplied by $k$, keeping $\eta / b$ and hence the noise level roughly constant, at least for moderate batch sizes.

> **Try it.** Re-run the cell with `lr=0.4` for batch 256 (the linear scaling rule applied from batch 64). Does the large-batch run catch up?

## 2. Optimizers from scratch and compared

Plain SGD has two well-known weaknesses. First, in a badly conditioned landscape (a long, narrow valley) the gradient points mostly across the valley rather than along it, so SGD zig-zags and creeps forward with a learning rate limited by the steepest direction. Second, one global learning rate must serve parameters whose gradients differ in scale by orders of magnitude (early layers versus late layers, frequent versus rare features). The optimizers of this section attack these two problems: **momentum methods** smooth the direction of travel, and **adaptive methods** give each parameter its own effective learning rate. We derive each update rule, implement it in a few lines using the base class above, and compare them all on two-dimensional test functions and on the MLP harness.

### 2.1 Momentum

**Intuition.** Imagine a heavy ball rolling down the loss surface. Its velocity accumulates the forces acting on it; components of the force that keep flipping sign (across the valley) cancel out, while components that point consistently in one direction (along the valley) add up. The ball moves faster along the valley and is damped across it.

**Update rule.** We keep a velocity $v$ (one buffer per parameter tensor) and update

$$
v_t \;=\; \beta\, v_{t-1} + g_t, \qquad \theta_t \;=\; \theta_{t-1} - \eta\, v_t,
$$

with the **momentum coefficient** $\beta \in [0, 1)$, almost always $0.9$. Unrolling the recursion shows what $v$ is:

$$
v_t \;=\; \sum_{k=1}^{t} \beta^{\,t-k} g_k
      \;=\; g_t + \beta g_{t-1} + \beta^2 g_{t-2} + \cdots,
$$

an exponentially weighted sum of past gradients with an effective memory of about $1/(1-\beta)$ steps (ten steps for $\beta = 0.9$). If the gradient were constant, $g_k = g$, the velocity would converge to $g/(1-\beta)$: momentum multiplies the effective step size by up to $1/(1-\beta) = 10$ along consistent directions, which is why the learning rate for momentum SGD is usually chosen smaller than for plain SGD. Across a valley, successive gradients alternate in sign and the sum stays small. Some texts write the EMA form $v_t = \beta v_{t-1} + (1-\beta) g_t$ instead; it is the same algorithm with the learning rate rescaled by $1/(1-\beta)$. PyTorch's `torch.optim.SGD(momentum=0.9)` uses the form above.

**Physics.** Discretizing a damped Newtonian particle $m\ddot\theta = -\nabla L - \gamma \dot\theta$ with a time step gives exactly this recurrence; $\beta$ plays the role of $1 - \gamma$, so $\beta = 0.9$ means ten percent of the velocity is lost to friction each step. The heavy-ball method is due to Polyak (1964); Sutskever et al. (2013) showed how important a well-tuned momentum is for deep networks.

### 2.2 Nesterov accelerated gradient

Momentum evaluates the gradient at the current point and then adds the velocity. Nesterov's variant (1983) first takes the momentum step and evaluates the gradient at the **look-ahead** point, which lets the ball "see" the slope ahead and correct itself before overshooting:

$$
v_t \;=\; \beta v_{t-1} + \nabla L\big(\theta_{t-1} - \eta \beta v_{t-1}\big), \qquad
\theta_t \;=\; \theta_{t-1} - \eta v_t .
$$

Evaluating the gradient at a shifted point is awkward in a framework where the gradient is computed at the current parameters. The standard trick is to change variables to the look-ahead point itself, $\phi_t = \theta_t - \eta\beta v_t$, and track $\phi$ instead of $\theta$. Substituting $\theta_{t-1} = \phi_{t-1} + \eta\beta v_{t-1}$ into the update and writing $g_t = \nabla L(\phi_{t-1})$ gives

$$
\phi_t \;=\; \theta_{t-1} - \eta(1+\beta) v_t
      \;=\; \phi_{t-1} + \eta\beta v_{t-1} - \eta(1+\beta)(\beta v_{t-1} + g_t)
      \;=\; \phi_{t-1} - \eta\,\big(g_t + \beta v_t\big),
$$

where the last equality uses $\beta^2 v_{t-1} + (1+\beta) g_t = \beta(\beta v_{t-1} + g_t) + g_t = \beta v_t + g_t$. So in terms of the tracked variable the update is: compute the gradient at the current point, update the velocity as in classical momentum, but step with $g_t + \beta v_t$ instead of $v_t$. This is exactly what `torch.optim.SGD(nesterov=True)` implements, and what our `Nesterov` class below does. In practice Nesterov momentum is a small but consistent improvement over classical momentum at no extra cost.

In [ ]:
class Momentum(Optimizer):
    """Heavy-ball momentum: v <- beta v + g;  theta <- theta - lr v"""
    def __init__(self, params, lr, beta=0.9):
        super().__init__(params, lr)
        self.beta = beta

    def update(self, p, g, state):
        v = state.setdefault("v", torch.zeros_like(p))
        v.mul_(self.beta).add_(g)
        p.sub_(self.lr * v)


class Nesterov(Momentum):
    """Nesterov momentum in the look-ahead parametrization: theta <- theta - lr (g + beta v)"""
    def update(self, p, g, state):
        v = state.setdefault("v", torch.zeros_like(p))
        v.mul_(self.beta).add_(g)
        p.sub_(self.lr * (g + self.beta * v))


# quick sanity check on a 1-D quadratic: momentum reaches the minimum faster than SGD
def steps_to_converge(opt_cls, lr, tol=1e-3, **kw):
    theta = torch.tensor([5.0], requires_grad=True)
    opt = opt_cls([theta], lr=lr, **kw)
    for t in range(1, 2001):
        loss = 0.5 * 0.05 * theta ** 2          # very flat bowl: curvature 0.05
        opt.zero_grad(); loss.sum().backward(); opt.step()
        if theta.abs().item() < tol:
            return t
    return None

for name, cls in [("SGD", SGD), ("Momentum", Momentum), ("Nesterov", Nesterov)]:
    print(f"{name:9s} lr=0.5 on a bowl with curvature 0.05: converged after {steps_to_converge(cls, 0.5)} steps")

On a flat bowl with curvature $0.05$ and learning rate $0.5$, gradient descent contracts the distance to the minimum by a factor $1 - 0.5 \cdot 0.05 = 0.975$ per step and needs several hundred steps. Momentum accumulates velocity in the consistent downhill direction, overshoots the minimum and spirals in, and arrives in a fraction of the steps: once the ball is oscillating, the heavy-ball method contracts the error by $\sqrt{\beta} \approx 0.95$ per step, twice as fast in the exponent as plain gradient descent, and the initial rush while the velocity builds up is faster still. This is the "along the valley" behaviour. In a real landscape both the flat and the steep directions are present at once, which is the situation we visualize in Section 2.7.

### 2.3 AdaGrad: per-parameter learning rates

**Intuition.** Suppose one weight receives large gradients at every step and another receives rare, small ones (think of a weight attached to a common input feature versus a rare one). A single learning rate is too large for the first and too small for the second. AdaGrad (Duchi, Hazan and Singer, 2011) scales each coordinate's step by the inverse root of its accumulated squared gradient:

$$
G_t \;=\; G_{t-1} + g_t \odot g_t, \qquad
\theta_t \;=\; \theta_{t-1} - \frac{\eta}{\sqrt{G_t} + \epsilon}\odot g_t ,
$$

where $\odot$ denotes element-wise multiplication, $G_0 = 0$, and $\epsilon \approx 10^{-8}$ avoids division by zero. Coordinates with a long history of large gradients get a small effective learning rate, rarely-updated coordinates a large one; the update is invariant to a rescaling of the gradient (multiplying all gradients by 10 changes nothing). The very first step has magnitude $\eta$ in every coordinate, so AdaGrad's learning rate has a different meaning from SGD's.

**Failure mode.** $G_t$ only grows, so the effective learning rate $\eta / \sqrt{G_t}$ decreases monotonically and eventually becomes so small that learning stops, whether or not the minimum has been reached. This is fine for convex problems (it is what the theory wants) but harmful for deep networks, and it is the problem RMSProp fixes.

### 2.4 RMSProp: forget the distant past

RMSProp (Tieleman and Hinton, 2012, from a Coursera lecture; never formally published) replaces the sum by an **exponential moving average** of the squared gradients so that the denominator tracks the *recent* gradient scale:

$$
s_t \;=\; \rho\, s_{t-1} + (1-\rho)\, g_t \odot g_t, \qquad
\theta_t \;=\; \theta_{t-1} - \frac{\eta}{\sqrt{s_t} + \epsilon}\odot g_t .
$$

With $\rho = 0.9$ the average has a memory of about ten steps; PyTorch's `torch.optim.RMSprop` uses $\rho = 0.99$ (called `alpha`) by default. Because $s_t$ is an average rather than a sum, it estimates $\mathbb{E}[g^2]$, and $\sqrt{s_t}$ is the root-mean-square gradient, hence the name. Each step then moves every coordinate by roughly $\eta$ times the *sign-like* normalized gradient $g_t / \text{RMS}(g)$, which makes the method robust to the scale of the loss and is why learning rates around $10^{-3}$ work across wildly different problems. A side effect is that near a minimum, where the gradients are tiny but their RMS is also tiny, the steps do not shrink on their own: RMSProp and Adam keep chattering with an amplitude of order $\eta$ until the learning rate is decayed (Section 3).

In [ ]:
class AdaGrad(Optimizer):
    """G <- G + g^2;  theta <- theta - lr g / (sqrt(G) + eps)"""
    def __init__(self, params, lr=0.01, eps=1e-8):
        super().__init__(params, lr)
        self.eps = eps

    def update(self, p, g, state):
        G = state.setdefault("G", torch.zeros_like(p))
        G.add_(g * g)
        p.sub_(self.lr * g / (G.sqrt() + self.eps))


class RMSProp(Optimizer):
    """s <- rho s + (1 - rho) g^2;  theta <- theta - lr g / (sqrt(s) + eps)"""
    def __init__(self, params, lr=1e-3, rho=0.9, eps=1e-8):
        super().__init__(params, lr)
        self.rho, self.eps = rho, eps

    def update(self, p, g, state):
        s = state.setdefault("s", torch.zeros_like(p))
        s.mul_(self.rho).add_((1 - self.rho) * g * g)
        p.sub_(self.lr * g / (s.sqrt() + self.eps))


# illustrate the scale invariance: multiplying the loss by 1000 does not change the RMSProp step
for scale in (1.0, 1000.0):
    theta = torch.tensor([1.0, -2.0], requires_grad=True)
    opt = RMSProp([theta], lr=0.1)
    loss = scale * (theta ** 2).sum()
    opt.zero_grad(); loss.backward()
    grad = theta.grad.tolist()
    opt.step()
    step = (theta.detach() - torch.tensor([1.0, -2.0])).tolist()
    print(f"loss scale {scale:7.1f}: gradient {grad} -> first step {[round(v, 4) for v in step]}")

The first RMSProp step is $-\eta \cdot g / (\sqrt{(1-\rho) g^2} + \epsilon) \approx -\eta\,\text{sign}(g)/\sqrt{1-\rho}$: independent of the size of the gradient, hence the identical steps for the two loss scales above (the factor $1/\sqrt{1-\rho} \approx 3.16$ is the same "cold start" bias that Adam corrects, as we see next).

### 2.5 Adam: momentum plus RMSProp, with bias correction

Adam (Kingma and Ba, 2015; the name stands for *adaptive moment estimation*) combines the two ideas: an exponential moving average of the gradient (the **first moment**, momentum) and of the squared gradient (the **second moment**, RMSProp), and it corrects both averages for their initialization bias. The full update is

$$
\begin{aligned}
m_t &= \beta_1 m_{t-1} + (1-\beta_1)\, g_t, &\qquad
s_t &= \beta_2 s_{t-1} + (1-\beta_2)\, g_t \odot g_t, \\
\hat m_t &= \frac{m_t}{1-\beta_1^{\,t}}, &\qquad
\hat s_t &= \frac{s_t}{1-\beta_2^{\,t}}, \\
\theta_t &= \theta_{t-1} - \eta\, \frac{\hat m_t}{\sqrt{\hat s_t} + \epsilon}. &&
\end{aligned}
$$

with defaults $\eta = 10^{-3}$, $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$, and $m_0 = s_0 = 0$.

**Deriving the bias correction.** Unroll the first-moment recursion as we did for momentum:

$$
m_t \;=\; (1-\beta_1)\sum_{k=1}^{t} \beta_1^{\,t-k}\, g_k .
$$

Suppose for a moment that the gradients are stationary with mean $\mathbb{E}[g]$. Taking expectations and summing the geometric series $\sum_{k=1}^{t}\beta_1^{t-k} = \sum_{j=0}^{t-1}\beta_1^{j} = \frac{1-\beta_1^{t}}{1-\beta_1}$ gives

$$
\mathbb{E}[m_t] \;=\; (1-\beta_1)\,\mathbb{E}[g]\,\frac{1-\beta_1^{t}}{1-\beta_1} \;=\; \big(1-\beta_1^{\,t}\big)\,\mathbb{E}[g].
$$

Because the average starts from zero, it is biased towards zero by the factor $(1-\beta_1^t)$, which is close to $0$ early on and tends to $1$ as $t \to \infty$. Dividing by that factor gives an unbiased estimate, $\mathbb{E}[\hat m_t] = \mathbb{E}[g]$. The same computation with $\beta_2$ gives $\mathbb{E}[s_t] = (1-\beta_2^t)\,\mathbb{E}[g^2]$ and the correction for $\hat s_t$.

**Why it matters.** With $\beta_2 = 0.999$, after the first step $s_1 = 0.001\, g_1^2$, so $\sqrt{s_1} \approx 0.03\,|g_1|$, while $m_1 = 0.1\, g_1$. Without correction the first update would be $\eta \cdot 0.1 / 0.03 \approx 3\eta$ in every coordinate, three times the intended size, and the second-moment estimate would stay too small for hundreds of steps (the memory of $\beta_2 = 0.999$ is a thousand steps), producing oversized steps exactly when the parameters are least settled. The next cell shows the numbers.

**Properties and typical use.** Because $\hat m_t / \sqrt{\hat s_t}$ is roughly of unit scale, each Adam step moves every parameter by at most about $\eta$ (a "trust region" per coordinate), which makes the default $\eta = 10^{-3}$ a remarkably safe starting point across architectures. Adam is the default choice for Transformers, generative models, reinforcement learning and most research code; a well-tuned SGD with momentum often generalizes slightly better on image classification with convolutional networks. Known failure modes: $\epsilon$ interacts with the gradient scale (for very small gradients, e.g. in some GANs or with mixed precision, $\epsilon$ should be raised to $10^{-6}$ or so); with $\beta_2$ close to $1$ Adam can fail to converge on adversarially constructed problems (Reddi et al., 2018 proposed AMSGrad, which keeps the maximum of past $s_t$); and the effective learning rate can spike early in training, which is one reason a **warmup** period (Section 3) is standard practice with Adam.

In [ ]:
b1, b2 = 0.9, 0.999
g = torch.tensor(1.0)                  # a constant gradient of size 1
m = s = torch.tensor(0.0)
print(" t   m_t     s_t      uncorrected m/sqrt(s)   corrected m_hat/sqrt(s_hat)")
for t in range(1, 6):
    m = b1 * m + (1 - b1) * g
    s = b2 * s + (1 - b2) * g * g
    m_hat, s_hat = m / (1 - b1 ** t), s / (1 - b2 ** t)
    print(f"{t:2d}  {m.item():.4f}  {s.item():.6f}   {(m / s.sqrt()).item():8.3f}                {(m_hat / s_hat.sqrt()).item():8.3f}")

For a constant gradient the correct normalized step is exactly $1$ (the gradient divided by its own RMS). The uncorrected ratio starts at $3.16$ and is still far from $1$ after five steps, whereas the bias-corrected ratio is $1$ from the very first step.

### 2.6 AdamW: decoupled weight decay

**L2 regularization** adds $\frac{\lambda}{2}\|\theta\|^2$ to the loss (Section 6 discusses why). Its gradient is $\lambda\theta$, so for plain SGD the update becomes

$$
\theta_t \;=\; \theta_{t-1} - \eta\,(g_t + \lambda\theta_{t-1}) \;=\; (1-\eta\lambda)\,\theta_{t-1} - \eta\, g_t ,
$$

which is why L2 regularization and **weight decay** (shrink the weights by a factor $1-\eta\lambda$, then take the gradient step) are used as synonyms: for SGD they are the same algorithm.

For Adam they are not. If we fold $\lambda\theta$ into the gradient, it passes through the adaptive normalization,

$$
\theta_t \;=\; \theta_{t-1} - \eta\,\frac{\widehat{m}_t\big[g + \lambda\theta\big]}{\sqrt{\hat s_t\big[g+\lambda\theta\big]} + \epsilon},
$$

so the regularization applied to a coordinate is divided by that coordinate's RMS gradient: weights with large, noisy gradients are barely regularized, weights with small gradients are strongly regularized, and the strength of the regularization is entangled with the learning rate and the second-moment estimate. Loshchilov and Hutter (2019) showed that this is why L2-regularized Adam generalizes worse than SGD, and proposed **decoupled weight decay**: compute the Adam step from the *unregularized* gradient and shrink the weights separately,

$$
\theta_t \;=\; \theta_{t-1} - \eta\left(\frac{\hat m_t}{\sqrt{\hat s_t}+\epsilon} + \lambda\,\theta_{t-1}\right).
$$

This is AdamW, today's default optimizer for Transformers, with typical $\lambda \in [0.01, 0.1]$ (much larger than the $\lambda \approx 5\cdot 10^{-4}$ used with SGD: the shrinkage per step is $\eta\lambda$ in both cases, and Adam's $\eta$ is about a hundred times smaller). The implementation below shares the Adam step between both classes; only where the decay enters differs.

In [ ]:
class Adam(Optimizer):
    """Adam with an optional L2 penalty folded into the gradient (the coupled, pre-AdamW form)."""
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0):
        super().__init__(params, lr)
        self.b1, self.b2 = betas
        self.eps, self.wd = eps, weight_decay

    def adam_step(self, p, g, state):
        m = state.setdefault("m", torch.zeros_like(p))
        s = state.setdefault("s", torch.zeros_like(p))
        m.mul_(self.b1).add_((1 - self.b1) * g)
        s.mul_(self.b2).add_((1 - self.b2) * g * g)
        m_hat = m / (1 - self.b1 ** self.t)
        s_hat = s / (1 - self.b2 ** self.t)
        p.sub_(self.lr * m_hat / (s_hat.sqrt() + self.eps))

    def update(self, p, g, state):
        if self.wd:
            g = g + self.wd * p                 # L2 penalty: goes through the adaptive scaling
        self.adam_step(p, g, state)


class AdamW(Adam):
    """Adam with decoupled weight decay: shrink the weights, then take the plain Adam step."""
    def update(self, p, g, state):
        if self.wd:
            p.mul_(1 - self.lr * self.wd)       # decay applied directly to the weights
        self.adam_step(p, g, state)


# the two forms of weight decay really differ for Adam. Two weights start at 10 and receive only
# zero-mean gradient noise (no signal from the data), with noise std 1 for the first and 100 for
# the second. Weight decay alone should pull both towards zero at the same rate.
noise_std = torch.tensor([1.0, 100.0])
for cls in (Adam, AdamW):
    torch.manual_seed(SEED)
    theta = torch.tensor([10.0, 10.0], requires_grad=True)
    opt = cls([theta], lr=0.1, weight_decay=0.1)
    for _ in range(1000):
        opt.zero_grad()
        theta.grad = torch.randn(2) * noise_std        # set the gradient directly: pure noise
        opt.step()
    print(f"{cls.__name__:5s}: theta after 1000 steps = {[round(v, 2) for v in theta.tolist()]}")

Both weights start at $10$ with the same decay strength $\lambda = 0.1$, and neither receives any useful gradient signal, so a sensible regularizer should shrink both towards zero. With the L2 penalty folded into the gradient (our `Adam` with `weight_decay`), the penalty $\lambda\theta$ is divided by the RMS gradient of its coordinate: for the low-noise weight the penalty dominates the denominator and the weight is driven to zero, but for the high-noise weight the penalty is a hundred times smaller than the noise, the normalized step is tiny, and after a thousand steps the weight has barely moved (its expected value is about $9$, give or take the few units of random walk that the noisy gradient adds). The regularization strength has become a function of gradient noise, which is not what anyone asked for. With decoupled decay (`AdamW`) both weights are multiplied by $1 - \eta\lambda = 0.99$ at every step regardless of their gradients, and both end up within the small noise band around zero ($0.99^{1000} \approx 4\cdot 10^{-5}$; the residual, typically a few tenths and at most about one, is the random walk driven by the noisy gradient).

### 2.7 Trajectories on two-dimensional test functions

To *see* the optimizers, we run them on two classic two-dimensional functions where we can draw the contours and the path of each method.

- A **badly conditioned quadratic** $f(x, y) = \tfrac12\left(x^2 + \kappa\, y^2\right)$ with $\kappa = 20$: a valley that is 20 times steeper in $y$ than in $x$. Gradient descent is stable only for $\eta < 2/\kappa = 0.1$, dictated by the steep direction, and then converges at rate $1 - \eta$ along the flat one. The **condition number** $\kappa$ is the ratio of the largest to the smallest curvature, and everything that makes plain gradient descent slow is captured by it.
- The **Rosenbrock function** $f(x, y) = (1-x)^2 + 100\,(y-x^2)^2$, whose minimum $(1, 1)$ lies at the bottom of a curved, extremely narrow valley. At the minimum the Hessian has eigenvalues of about $1000$ (across the valley) and $0.4$ (along its floor), a condition number around $2500$; worse, the valley bends, so a fixed direction of travel is never right for long.

The learning rates below were chosen per method (each family reads $\eta$ differently: the first AdaGrad and Adam steps have size $\eta$ per coordinate and RMSProp's about $3\eta$, whereas an SGD step is $\eta$ times the gradient). The helper stops early if a run diverges.

In [ ]:
def quad(x, y, kappa=20.0):
    """Badly conditioned bowl: curvature 1 along x, kappa along y. Works on tensors and NumPy arrays."""
    return 0.5 * (x ** 2 + kappa * y ** 2)


def rosenbrock(x, y):
    return (1 - x) ** 2 + 100 * (y - x ** 2) ** 2


def run_2d(fn, opt_cls, start, steps, **kw):
    """Optimize fn(x, y) from `start` with one of our optimizers; return the path as a (steps+1, 2) array."""
    theta = torch.tensor(start, dtype=torch.float32, requires_grad=True)
    opt = opt_cls([theta], **kw)
    path = [theta.detach().clone()]
    for _ in range(steps):
        loss = fn(theta[0], theta[1])
        opt.zero_grad(); loss.backward(); opt.step()
        if not torch.isfinite(theta).all():
            break
        path.append(theta.detach().clone())
    return torch.stack(path).cpu().numpy()    # (these are CPU tensors already; .cpu() is a harmless habit)


def plot_paths(fn, paths, xlim, ylim, levels, title, minimum):
    xs, ys = np.linspace(*xlim, 300), np.linspace(*ylim, 300)
    XX, YY = np.meshgrid(xs, ys)
    plt.figure(figsize=(8.5, 5))
    plt.contour(XX, YY, fn(XX, YY), levels=levels, cmap="Greys", linewidths=0.7)
    for name, p in paths.items():
        plt.plot(p[:, 0], p[:, 1], ".-", ms=3, lw=1, label=f"{name}: f = {fn(p[-1, 0], p[-1, 1]):.2e}")
    plt.plot(*minimum, "r*", ms=13, label="minimum")
    plt.xlim(xlim); plt.ylim(ylim); plt.xlabel("x"); plt.ylabel("y"); plt.title(title)
    plt.legend(fontsize=8, loc="upper left"); plt.tight_layout(); plt.show()


runs_quad = {
    "SGD lr=0.09": (SGD, dict(lr=0.09)),
    "Momentum lr=0.02": (Momentum, dict(lr=0.02)),
    "Nesterov lr=0.02": (Nesterov, dict(lr=0.02)),
    "AdaGrad lr=1.0": (AdaGrad, dict(lr=1.0)),
    "RMSProp lr=0.2": (RMSProp, dict(lr=0.2)),
    "Adam lr=0.3": (Adam, dict(lr=0.3)),
}
paths_quad = {name: run_2d(quad, cls, (-9.0, 2.0), steps=100, **kw) for name, (cls, kw) in runs_quad.items()}
plot_paths(quad, paths_quad, (-10, 4), (-3, 3), levels=np.linspace(0, 9, 19) ** 2,
           title="Badly conditioned quadratic (kappa = 20), 100 steps each", minimum=(0, 0))

Read the figure method by method. **SGD** at $\eta = 0.09$, just below the stability limit, bounces violently across the steep $y$ direction (the multiplier per step is $1 - 0.09 \cdot 20 = -0.8$: overshoot, shrink by 20%, overshoot again) while creeping along $x$ with multiplier $0.91$; a smaller learning rate would remove the bouncing but slow the creeping further. This coupling is the curse of conditioning. **Momentum** and **Nesterov** use a learning rate more than four times smaller and still arrive first: velocity accumulates along $x$ where gradients are consistent, while the alternating $y$ gradients cancel. They overshoot the minimum and spiral in, and Nesterov's look-ahead damps the spiral slightly faster. **AdaGrad** takes a large first step and then progressively smaller ones as $G$ accumulates. **RMSProp** and **Adam** move in steps of roughly $\eta$ per coordinate regardless of the gradient magnitude, so they walk to the valley floor in a few steps and then along it in a nearly straight line; note the chatter near the minimum, where the steps do not shrink because the normalized gradient always has unit size.

In [ ]:
runs_rb = {
    "SGD lr=1e-3": (SGD, dict(lr=1e-3)),
    "Momentum lr=1e-3": (Momentum, dict(lr=1e-3)),
    "Nesterov lr=1e-3": (Nesterov, dict(lr=1e-3)),
    "AdaGrad lr=0.3": (AdaGrad, dict(lr=0.3)),
    "RMSProp lr=0.01": (RMSProp, dict(lr=0.01)),
    "Adam lr=0.02": (Adam, dict(lr=0.02)),
}
STEPS_RB = 500
paths_rb = {name: run_2d(rosenbrock, cls, (-1.2, 1.0), steps=STEPS_RB, **kw) for name, (cls, kw) in runs_rb.items()}
plot_paths(rosenbrock, paths_rb, (-2, 2), (-1, 3), levels=np.logspace(-0.5, 3.5, 20),
           title=f"Rosenbrock function, {STEPS_RB} steps each", minimum=(1, 1))
for name, p in paths_rb.items():
    print(f"{name:18s} final point ({p[-1, 0]:7.3f}, {p[-1, 1]:7.3f})  distance to minimum {np.hypot(p[-1, 0] - 1, p[-1, 1] - 1):.3f}")

On the Rosenbrock function the picture changes. **SGD** must use $\eta = 10^{-3}$ to survive the curvature of the valley walls, and with that learning rate it reaches the valley floor and then barely moves along it in 500 steps. **Momentum** and **Nesterov** get much further along the floor, but their velocity keeps carrying them into the walls, so the path oscillates across the valley as it curves. The adaptive methods take steps of roughly constant size per coordinate: **AdaGrad** starts fast and then slows as its accumulator grows, whereas **RMSProp** and **Adam** follow the bend of the valley, since the sign-like normalized gradient re-orients them at every step. Adam's momentum smooths RMSProp's zig-zag, but at this learning rate it advances along the floor more slowly: with $\beta_2 = 0.999$ its second-moment estimate still remembers the gradients of size $\sim 200$ from the first steps, so for hundreds of steps its normalized step stays far below $\eta$ (RMSProp, with $\rho = 0.9$, forgets them within about ten steps). The printed distances show how far each method got; with a constant learning rate none of them settles exactly on the minimum (RMSProp and Adam keep chattering with an amplitude of order $\eta$), and none of these two-dimensional pictures should be over-interpreted: the loss of a neural network is far higher-dimensional and noisier. But the mechanisms they show (zig-zag from conditioning, velocity along consistent directions, per-coordinate normalization) are exactly the ones at work in high dimension.

> **Try it.** Increase the SGD learning rate on the Rosenbrock function to `1e-2`. The run blows up within a few steps (the helper stops when the parameters become infinite). Then try `2e-3`: it survives, although the largest curvature of the Hessian at the starting point is about $1500$ and the quadratic stability limit $2/\lambda_{\max}$ would be around $1.3\cdot 10^{-3}$. The first overshoot lands in a region of lower curvature, a reminder that the linear stability analysis is only local on a non-quadratic function.

### 2.8 Comparison on the FashionMNIST harness

Two-dimensional pictures build intuition; the real test is a network. We now train the same MLP from the same initial weights with each of the seven optimizers, using their conventional learning rates (plain SGD needs a large one, the momentum methods several times smaller (here one fifth) because of the $1/(1-\beta)$ amplification, and the adaptive methods their usual $10^{-3}$, or $10^{-2}$ for AdaGrad, whose steps shrink over time). Everything else, including the mini-batch order, is identical across runs.

In [ ]:
optim_configs = {
    "SGD lr=0.1": (SGD, dict(lr=0.1)),
    "Momentum lr=0.02": (Momentum, dict(lr=0.02)),
    "Nesterov lr=0.02": (Nesterov, dict(lr=0.02)),
    "AdaGrad lr=0.01": (AdaGrad, dict(lr=0.01)),
    "RMSProp lr=1e-3": (RMSProp, dict(lr=1e-3)),
    "Adam lr=1e-3": (Adam, dict(lr=1e-3)),
    "AdamW lr=1e-3 wd=0.01": (AdamW, dict(lr=1e-3, weight_decay=0.01)),
}
optim_results = {}
for name, (cls, kw) in optim_configs.items():
    model = make_mlp()
    opt = cls(model.parameters(), **kw)
    t0 = time.time()
    h = train(model, opt, EPOCHS, verbose=False)
    optim_results[name] = h
    print(f"{name:24s} | train loss {h['train_loss'][-1]:.4f} | val loss {h['val_loss'][-1]:.4f} | "
          f"val acc {h['val_acc'][-1]:.3f} | {time.time() - t0:4.1f}s")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for name, h in optim_results.items():
    ax1.plot(smooth(h["step_loss"]), lw=1, label=name)
    ax2.plot(range(1, EPOCHS + 1), h["val_acc"], "o-", ms=4, label=name)
ax1.set_xlabel("optimizer step"); ax1.set_ylabel("training loss (running mean of 20 steps)")
ax1.set_title("Training loss per step"); ax1.set_yscale("log"); ax1.legend(fontsize=7)
ax2.set_xlabel("epoch"); ax2.set_ylabel("validation accuracy"); ax2.set_title("Validation accuracy per epoch")
ax2.legend(fontsize=7)
plt.tight_layout(); plt.show()

The typical outcome (details depend on the budget and the seed) is that the momentum methods and Adam/AdamW reach a low training loss fastest, plain SGD is slower but steady, RMSProp behaves like Adam without the smoothing of momentum, and AdaGrad starts quickly and then slows down as its accumulated denominators grow. On this easy, well-conditioned problem the *final* validation accuracies are close, which is itself a lesson: with a sensible learning rate most optimizers converge to comparable solutions on small problems, and the differences that matter in practice show up in how much tuning each requires, how robust it is to the learning rate, and how it behaves on hard, badly scaled problems such as Transformers. AdamW's small weight decay changes little in a few epochs; its benefit appears in long runs where unregularized weights would drift to large norms.

### 2.9 The equivalent `torch.optim` calls

You will not use your own optimizer classes in real projects: `torch.optim` implements the same algorithms with fused, multi-tensor kernels and support for parameter groups (different learning rates for different parts of a model) and state saving. But it is worth verifying that our implementations really are the same algorithms. The next cell trains two identical models for twenty steps on the same mini-batch, one with our class and one with the PyTorch counterpart, and reports the largest difference between the resulting parameters. The only subtleties are default hyperparameters that differ between libraries (`Adagrad` uses $\epsilon = 10^{-10}$, `RMSprop` uses $\rho = 0.99$ under the name `alpha`), which we set explicitly.

In [ ]:
def max_param_diff(my_cls, my_kw, torch_cls, torch_kw, steps=20):
    """Train two identical models with our optimizer and its torch.optim twin; return the max parameter gap."""
    m1, m2 = make_mlp(), make_mlp()                         # identical initial weights (same seed)
    o1 = my_cls(m1.parameters(), **my_kw)
    o2 = torch_cls(m2.parameters(), **torch_kw)
    xb, yb = X_train[:BATCH], y_train[:BATCH]
    for _ in range(steps):
        for m, o in ((m1, o1), (m2, o2)):
            o.zero_grad()
            F.cross_entropy(m(xb), yb).backward()
            o.step()
    return max((p1 - p2).abs().max().item() for p1, p2 in zip(m1.parameters(), m2.parameters()))


pairs = [
    ("SGD", SGD, dict(lr=0.1), torch.optim.SGD, dict(lr=0.1)),
    ("Momentum", Momentum, dict(lr=0.02), torch.optim.SGD, dict(lr=0.02, momentum=0.9)),
    ("Nesterov", Nesterov, dict(lr=0.02), torch.optim.SGD, dict(lr=0.02, momentum=0.9, nesterov=True)),
    ("AdaGrad", AdaGrad, dict(lr=0.01, eps=1e-10), torch.optim.Adagrad, dict(lr=0.01, eps=1e-10)),
    ("RMSProp", RMSProp, dict(lr=1e-3, rho=0.99), torch.optim.RMSprop, dict(lr=1e-3, alpha=0.99)),
    ("Adam", Adam, dict(lr=1e-3), torch.optim.Adam, dict(lr=1e-3)),
    ("AdamW", AdamW, dict(lr=1e-3, weight_decay=0.01), torch.optim.AdamW, dict(lr=1e-3, weight_decay=0.01)),
]
for name, c1, k1, c2, k2 in pairs:
    print(f"{name:9s} max |theta_ours - theta_torch| after 20 steps: {max_param_diff(c1, k1, c2, k2):.1e}")

The gaps are at the level of float32 round-off ($10^{-7}$ or so; the libraries perform the same arithmetic in a slightly different order), which confirms that the classes above are faithful implementations. From here on we use `torch.optim` whenever we need an optimizer and keep our own classes for experiments that need access to the internals.

**Typical hyperparameters and when to use which.** The following table summarizes common practice; treat it as a starting point for tuning, not as a law.

| Optimizer | `torch.optim` | Typical settings | Use it when |
|---|---|---|---|
| SGD | `SGD(lr=0.1)` | $\eta \in [0.01, 0.5]$ | teaching, convex problems, as a baseline |
| SGD + (Nesterov) momentum | `SGD(lr=0.05, momentum=0.9, nesterov=True)` | $\eta \in [0.01, 0.1]$, $\beta = 0.9$, weight decay $5\cdot 10^{-4}$ | convolutional networks for image classification; often the best final accuracy when tuned |
| AdaGrad | `Adagrad(lr=0.01)` | $\eta \in [0.01, 0.1]$ | sparse features (embeddings, bag-of-words); rarely for deep nets |
| RMSProp | `RMSprop(lr=1e-3, alpha=0.99)` | $\eta \in [10^{-4}, 10^{-2}]$ | recurrent networks, reinforcement learning (historical) |
| Adam | `Adam(lr=1e-3, betas=(0.9, 0.999))` | $\eta \in [10^{-4}, 3\cdot 10^{-3}]$ | the safe default for almost everything |
| AdamW | `AdamW(lr=1e-3, weight_decay=0.01)` | $\lambda \in [0.01, 0.1]$, $\beta_2 = 0.95$ for large language models | Transformers, any model where you want weight decay with Adam |

Two practical notes. First, learning rates are not transferable between families: an SGD learning rate of $0.1$ and an Adam learning rate of $10^{-3}$ are equally "normal". Second, weight decay should usually not be applied to biases and normalization parameters; this is done with parameter groups, which we use in Notebook 05 · The PyTorch Workflow and Best Practices and in Notebook 09 · Attention and Transformers.

> **Going deeper.** Newer optimizers keep appearing: Lion (Chen et al., 2023) uses only the sign of a momentum term and needs a learning rate three to ten times smaller than Adam's; Adafactor and 8-bit Adam reduce optimizer memory (Adam stores two extra tensors per parameter, tripling the memory of the parameters themselves; see Notebook 14 · Efficient Training and Scaling); Shampoo and Muon use matrix-shaped preconditioners. Nadam (Adam with Nesterov momentum) is Exercise 2.

## 3. Learning-rate schedules

### 3.1 Why the learning rate should change during training

The learning rate is the most important hyperparameter, and no single value is right for the whole run. Early on we are far from any minimum and want large steps; late in training the mini-batch noise (of size proportional to $\eta$) keeps the parameters bouncing around inside a "noise ball" whose radius shrinks only when $\eta$ does. Section 2 also showed that RMSProp and Adam do not reduce their step size near a minimum by themselves. A **learning-rate schedule** $\eta_t$ therefore decays the learning rate over the run; this is a form of **annealing**, and for SGD on convex problems the classical theory requires $\sum_t \eta_t = \infty$ and $\sum_t \eta_t^2 < \infty$ (for example $\eta_t \propto 1/t$) to converge to the exact minimum.

Writing $\eta_0$ for the initial learning rate, $t$ for the step, and $T$ for the total number of steps, the schedules in common use are:

- **Constant**: $\eta_t = \eta_0$. The baseline.
- **Step decay**: $\eta_t = \eta_0 \gamma^{\lfloor t / s\rfloor}$, e.g. multiply by $\gamma = 0.1$ every $s$ steps (the classic ImageNet recipe: divide by 10 at epochs 30, 60, 90).
- **Exponential decay**: $\eta_t = \eta_0 \gamma^t$ with $\gamma$ just below 1.
- **Cosine annealing** (Loshchilov and Hutter, 2017): $\eta_t = \eta_{\min} + \tfrac12(\eta_0 - \eta_{\min})\big(1 + \cos(\pi t / T)\big)$. Smooth, no milestones to tune, spends a long time at high learning rates and ends at $\eta_{\min}$; the default in most modern recipes.
- **Linear warmup + cosine**: $\eta_t = \eta_0\, t / T_w$ for $t < T_w$, then cosine decay. Warmup (typically $1$–$5\%$ of the run) protects the early steps, when the parameters and Adam's second-moment estimates are unreliable and a full-size learning rate can throw the network into a bad region; it is standard for Transformers and for large batches.
- **One-cycle** (Smith and Topin, 2019): ramp from $\eta_{\max}/25$ up to the peak $\eta_{\max}$ over the first 30% of the run, then anneal down to almost zero (these are the defaults of PyTorch's `OneCycleLR`); often paired with an inverse cycle for the momentum. Good for fast training under a fixed budget.
- **ReduceLROnPlateau**: not a function of $t$ at all, but of the validation loss: divide $\eta$ by a factor when the validation loss has not improved for a number of epochs. Convenient when the run length is unknown.

In PyTorch a scheduler wraps an optimizer and overwrites `param_groups[i]["lr"]` on every call of `scheduler.step()`. Whether that call is made once per epoch or once per mini-batch is up to you, as long as the schedule's horizon is measured in the same unit. We step per mini-batch, which gives smooth curves and is what warmup needs. The `LambdaLR` scheduler takes an arbitrary function of the step count and multiplies the base learning rate by it, which is the easiest way to implement warmup. Let us draw all of them.

In [ ]:
from torch.optim.lr_scheduler import StepLR, ExponentialLR, CosineAnnealingLR, LambdaLR, OneCycleLR


def warmup_cosine(warmup_steps, total_steps, min_frac=0.0):
    """Multiplicative factor for LambdaLR: linear warmup to 1, then cosine decay to min_frac."""
    def factor(step):
        if step < warmup_steps:
            return (step + 1) / warmup_steps
        progress = min(1.0, (step - warmup_steps) / max(1, total_steps - warmup_steps))
        return min_frac + (1 - min_frac) * 0.5 * (1 + math.cos(math.pi * progress))
    return factor


def schedule_curve(make_sched, total_steps, lr0=0.1):
    """Record the learning rate that a scheduler produces over total_steps optimizer steps."""
    p = torch.zeros(1, requires_grad=True)
    opt = torch.optim.SGD([p], lr=lr0)
    sched = make_sched(opt)
    lrs = []
    for _ in range(total_steps):
        lrs.append(opt.param_groups[0]["lr"])
        opt.step()                              # (no gradient: the step is a no-op)
        if sched is not None:
            sched.step()
    return lrs


T_DEMO = 1000
schedules_demo = {
    "constant": lambda o: None,
    "step (x0.1 every 300)": lambda o: StepLR(o, step_size=300, gamma=0.1),
    "exponential (0.997^t)": lambda o: ExponentialLR(o, gamma=0.997),
    "cosine": lambda o: CosineAnnealingLR(o, T_max=T_DEMO),
    "warmup 10% + cosine": lambda o: LambdaLR(o, warmup_cosine(100, T_DEMO)),
    "one-cycle": lambda o: OneCycleLR(o, max_lr=0.1, total_steps=T_DEMO),
}
plt.figure(figsize=(8, 4.5))
for name, mk in schedules_demo.items():
    plt.plot(schedule_curve(mk, T_DEMO), label=name)
plt.xlabel("optimizer step"); plt.ylabel("learning rate")
plt.title("Learning-rate schedules over 1000 steps (base lr = 0.1)")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

The plot shows the shapes described above over a run of 1000 steps: the staircase of the step schedule, the smooth geometric decay of the exponential one, the slow start and fast finish of the cosine, the linear ramp of the warmup over the first 100 steps, and the one-cycle rise from $\eta_{\max}/25$ to the peak at 30% of the run followed by the anneal to almost zero. Only the constant schedule ends where it started; every other one finishes at a small fraction of the base learning rate.

### 3.2 The learning-rate range test

How do we choose $\eta_0$ in the first place? The **learning-rate range test** (Smith, 2017) is a cheap, one-run answer: starting from a tiny learning rate, increase it geometrically every mini-batch while recording the training loss. The loss first stays flat (the learning rate is too small to move), then falls (the useful range), then explodes (too large). A good choice is somewhere on the descending part, typically three to ten times smaller than the learning rate where the loss is lowest, since the lowest point is already on the edge of instability. We smooth the recorded losses with a bias-corrected exponential moving average (the same formula as Adam's $\hat m_t$) and stop when the smoothed loss exceeds four times its best value.

In [ ]:
def lr_range_test(model, opt_cls, lr_min=1e-5, lr_max=1.0, steps=150, batch_size=BATCH, beta=0.98, **opt_kw):
    """Increase the learning rate geometrically from lr_min to lr_max, one mini-batch per step.

    Works with our own optimizer classes (they expose a writable `lr`). Returns (lrs, smoothed losses).
    """
    opt = opt_cls(model.parameters(), lr=lr_min, **opt_kw)
    gamma = (lr_max / lr_min) ** (1.0 / steps)
    gen = torch.Generator().manual_seed(SEED)
    lrs, losses, avg, best = [], [], 0.0, float("inf")
    model.train()
    for i in range(steps):
        idx = torch.randint(0, len(X_train), (batch_size,), generator=gen).to(device)
        loss = F.cross_entropy(model(X_train[idx]), y_train[idx])
        opt.zero_grad(); loss.backward(); opt.step()
        avg = beta * avg + (1 - beta) * loss.item()
        smoothed = avg / (1 - beta ** (i + 1))            # bias-corrected EMA of the loss
        if not math.isfinite(smoothed) or smoothed > 4 * best:
            break                                          # diverging: stop the test
        lrs.append(opt.lr); losses.append(smoothed)
        best = min(best, smoothed)
        opt.lr *= gamma
    return lrs, losses


RANGE_STEPS = 150 if FAST_RUN else 300
range_lrs, range_losses = lr_range_test(make_mlp(), Nesterov, steps=RANGE_STEPS)
best_i = int(np.argmin(range_losses))
plt.figure()
plt.semilogx(range_lrs, range_losses)
plt.axvline(range_lrs[best_i], color="C3", ls="--", label=f"loss minimum at lr = {range_lrs[best_i]:.2g}")
plt.axvline(range_lrs[best_i] / 5, color="C2", ls="--", label=f"suggested lr = {range_lrs[best_i] / 5:.2g}")
plt.xlabel("learning rate (log scale)"); plt.ylabel("smoothed training loss")
plt.title("Learning-rate range test, Nesterov momentum"); plt.legend()
plt.tight_layout(); plt.show()
print(f"stopped after {len(range_lrs)} steps at lr = {range_lrs[-1]:.2g}")

The curve has the expected three regimes. The flat start tells us that learning rates below about $10^{-3}$ are useless for this optimizer; the steep descent around $10^{-2}$ to $10^{-1}$ is where training is efficient; and shortly after the minimum the loss blows up. The suggested learning rate (one fifth of the minimum's) is close to the $0.02$ we used for the momentum methods in Section 2, which is reassuring. Exercise 5 turns this into a reusable tool.

### 3.3 Experiment: schedules compared at a fixed budget

We now train the MLP with Nesterov momentum under five schedules and the same total number of steps. Every schedule uses the same base learning rate $\eta_0 = 0.05$, except one-cycle, whose *peak* is set to $2\eta_0$ because it spends most of its time below the peak.

In [ ]:
steps_per_epoch = math.ceil(N_TRAIN / BATCH)
T_TOTAL = EPOCHS * steps_per_epoch
LR0 = 0.05
sched_configs = {
    "constant": lambda o: None,
    "step decay (x0.1 every 40%)": lambda o: StepLR(o, step_size=max(1, int(0.4 * T_TOTAL)), gamma=0.1),
    "cosine": lambda o: CosineAnnealingLR(o, T_max=T_TOTAL),
    "warmup 10% + cosine": lambda o: LambdaLR(o, warmup_cosine(max(1, int(0.1 * T_TOTAL)), T_TOTAL)),
    "one-cycle (peak 2x)": lambda o: OneCycleLR(o, max_lr=2 * LR0, total_steps=T_TOTAL),
}
sched_results = {}
for name, mk in sched_configs.items():
    model = make_mlp()
    opt = torch.optim.SGD(model.parameters(), lr=LR0, momentum=0.9, nesterov=True)
    h = train(model, opt, EPOCHS, scheduler=mk(opt), verbose=False)
    sched_results[name] = h
    print(f"{name:26s} | final train loss {h['train_loss'][-1]:.4f} | val loss {h['val_loss'][-1]:.4f} | "
          f"val acc {h['val_acc'][-1]:.3f} | final lr {h['lr'][-1]:.1e}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for name, h in sched_results.items():
    ax1.plot(smooth(h["step_loss"]), lw=1, label=name)
    ax2.plot(range(1, EPOCHS + 1), h["val_loss"], "o-", ms=4, label=name)
ax1.set_xlabel("optimizer step"); ax1.set_ylabel("training loss (smoothed)"); ax1.set_title("Training loss")
ax1.set_yscale("log"); ax1.legend(fontsize=7)
ax2.set_xlabel("epoch"); ax2.set_ylabel("validation loss"); ax2.set_title("Validation loss per epoch")
ax2.legend(fontsize=7)
plt.tight_layout(); plt.show()

Look at the end of the training-loss curves: the schedules that decay to (almost) zero finish with a visibly lower and smoother loss than the constant schedule, because the noise ball has shrunk with the learning rate; the constant run is still bouncing. The step schedule shows the characteristic drops at its milestones (40% and 80% of the run). Under `FAST_RUN` the budget is tiny and the validation differences are within noise; in the full run the decaying schedules typically end with a lower validation loss and a slightly higher accuracy than the constant one, and the warmup makes no difference for an MLP with momentum SGD (it matters for Adam and for deep Transformers). The general recommendation is simple: always decay the learning rate towards the end of training, use cosine (with a short warmup if you use Adam) unless you have a reason not to, and set the horizon $T$ to the length of the run you actually intend to do, since a cosine schedule that is interrupted halfway is just a slowly decaying constant.

> **Common pitfall.** Forgetting to call `scheduler.step()` (the learning rate stays constant), or calling it per epoch when the scheduler was configured in steps (the decay never happens). Print or log `optimizer.param_groups[0]["lr"]` at every epoch, as our `train` function does.

## 4. Initialization

### 4.1 Why initialization matters: variance propagation

Consider a linear layer $y = Wx$ with $W \in \mathbb{R}^{n_{\text{out}} \times n_{\text{in}}}$, no bias for now. Suppose the entries $W_{ij}$ are i.i.d. with mean zero and variance $\sigma_W^2$, and the inputs $x_j$ are i.i.d. with mean zero and variance $\sigma_x^2$, independent of $W$. Then each output

$$
y_i = \sum_{j=1}^{n_{\text{in}}} W_{ij} x_j
$$

is a sum of $n_{\text{in}}$ independent zero-mean terms with variance $\sigma_W^2 \sigma_x^2$ each, so

$$
\operatorname{Var}(y_i) \;=\; n_{\text{in}}\, \sigma_W^2\, \sigma_x^2 .
$$

Stack $L$ such layers (ignoring the non-linearity for a moment) and the variance of the signal is multiplied by $n_{\text{in}} \sigma_W^2$ at each of them: after $L$ layers it is scaled by $(n_{\text{in}} \sigma_W^2)^L$. Unless that factor is exactly $1$, the activations either vanish or explode exponentially with depth. Keeping the forward variance constant therefore requires

$$
\sigma_W^2 \;=\; \frac{1}{n_{\text{in}}} \qquad \text{(the "fan-in" condition)}.
$$

The same argument applies to the backward pass. Backpropagation through the layer computes $\partial L/\partial x = W^\top\, \partial L/\partial y$, a sum over $n_{\text{out}}$ terms, so the gradient variance is scaled by $n_{\text{out}} \sigma_W^2$ per layer, and keeping gradients well-scaled requires $\sigma_W^2 = 1/n_{\text{out}}$ (the "fan-out" condition). Both cannot hold unless $n_{\text{in}} = n_{\text{out}}$, and Glorot and Bengio (2010) proposed the compromise

$$
\sigma_W^2 \;=\; \frac{2}{n_{\text{in}} + n_{\text{out}}} \qquad \text{(Xavier / Glorot initialization)},
$$

usually sampled from a uniform distribution $\mathcal{U}(-a, a)$ with $a = \sqrt{6/(n_{\text{in}}+n_{\text{out}})}$ (a uniform on $[-a, a]$ has variance $a^2/3$) or from a normal with that variance. This analysis assumes an activation function that behaves like the identity around zero, which is true for $\tanh$ when the pre-activations are small (the sigmoid has an offset of $1/2$ and a slope of only $1/4$ at zero, so it fits the analysis poorly; Section 4.3 shows the consequence).

**ReLU changes the constant.** A ReLU sets half of its inputs to zero. If $z$ is symmetric around zero, $\mathbb{E}[\mathrm{ReLU}(z)^2] = \tfrac12 \mathbb{E}[z^2]$: the second moment is halved at every layer. To compensate, He et al. (2015) doubled the weight variance:

$$
\sigma_W^2 \;=\; \frac{2}{n_{\text{in}}} \qquad \text{(He / Kaiming initialization)}.
$$

With it, a ReLU network of any depth keeps its activations at a constant scale at initialization, which is what allowed the very deep networks of the ResNet era to be trained (together with normalization and skip connections; see Notebook 06 · Convolutional Neural Networks). PyTorch's `nn.Linear` default is `kaiming_uniform_(a=sqrt(5))`, which works out to $\mathcal{U}(-1/\sqrt{n_{\text{in}}}, 1/\sqrt{n_{\text{in}}})$, i.e. variance $1/(3 n_{\text{in}})$: smaller than both recipes, chosen for backward compatibility, and adequate for shallow networks. For deep plain networks you should set the initialization explicitly with `nn.init.kaiming_normal_` or `xavier_uniform_`, as we do below with `model.apply`.

Two further points. Biases are initialized to zero. And **symmetry breaking** is why the weights must be random at all: if all weights of a layer were equal (for instance all zero), every unit in the layer would compute the same function, receive the same gradient, and remain identical forever; the layer would effectively have one unit.

### 4.2 Experiment: activations and gradients through a 10-layer network

The cell below builds a 10-layer network of width 256 with a chosen activation and initialization, pushes one batch through it, backpropagates a cross-entropy loss from a small output head, and records the standard deviation of the activations and the norm of the weight gradient at every layer. We compare five initializations: all zeros, too small ($\sigma_W = 0.01$), too large ($\sigma_W = 0.2$), Xavier and He, for both $\tanh$ and ReLU networks.

In [ ]:
def layer_statistics(init_std, act, depth=10, width=256, n_batch=512, in_dim=784):
    """One forward/backward pass through a deep MLP at initialization.

    init_std(fan_in, fan_out) gives the std of the normal distribution used for each weight matrix.
    Returns per-layer activation std, per-layer gradient norm of W, and the activations themselves.
    """
    torch.manual_seed(SEED)
    dims = [in_dim] + [width] * depth
    layers = []
    for i in range(depth):
        lin = nn.Linear(dims[i], dims[i + 1], bias=False).to(device)
        std = init_std(dims[i], dims[i + 1])
        if std > 0:
            nn.init.normal_(lin.weight, std=std)
        else:
            nn.init.zeros_(lin.weight)
        layers.append(lin)
    head = nn.Linear(width, 10).to(device)
    x, y = X_train[:n_batch], y_train[:n_batch]
    acts, h = [], x
    for lin in layers:
        h = act(lin(h))
        acts.append(h)
    loss = F.cross_entropy(head(h), y)
    loss.backward()
    act_std = [a.std().item() for a in acts]
    grad_norm = [lin.weight.grad.norm().item() for lin in layers]
    return act_std, grad_norm, [a.detach().cpu().numpy() for a in acts]


inits = {
    "zero": lambda fi, fo: 0.0,
    "small (std 0.01)": lambda fi, fo: 0.01,
    "large (std 0.2)": lambda fi, fo: 0.2,
    "Xavier": lambda fi, fo: math.sqrt(2.0 / (fi + fo)),
    "He": lambda fi, fo: math.sqrt(2.0 / fi),
}
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for row, (act_name, act) in enumerate([("tanh", torch.tanh), ("ReLU", F.relu)]):
    for name, init_std in inits.items():
        a_std, g_norm, _ = layer_statistics(init_std, act)
        xs = np.arange(1, len(a_std) + 1)
        axes[row, 0].semilogy(xs, np.maximum(a_std, 1e-12), "o-", ms=3, label=name)
        axes[row, 1].semilogy(xs, np.maximum(g_norm, 1e-12), "o-", ms=3, label=name)
    axes[row, 0].set_title(f"{act_name} network: activation std per layer")
    axes[row, 1].set_title(f"{act_name} network: gradient norm of W per layer")
    axes[row, 0].set_ylabel("std of activations (log)"); axes[row, 1].set_ylabel("||dL/dW|| (log)")
for ax in axes.flat:
    ax.set_xlabel("layer"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

Every prediction of the variance analysis is visible (values below $10^{-12}$ are clipped to that floor for the log axis):

- **Zero** initialization produces zero activations and zero gradients in every hidden layer: nothing can be learned, because of the symmetry argument.
- **Too small** ($\sigma_W = 0.01$, so $n_{\text{in}}\sigma_W^2 \approx 0.026$ for width 256): the activation scale shrinks by a factor of about $6$ per layer and is around $10^{-8}$ after ten layers; the gradients of the early layers are correspondingly tiny. The network is not dead, but it is frozen.
- **Too large** ($\sigma_W = 0.2$, $n_{\text{in}}\sigma_W^2 \approx 10$): in the $\tanh$ network the activations saturate at $\pm 1$ (the std settles near $1$, which looks healthy but is not, see the histograms below) and the gradients vanish because $\tanh'(z) \approx 0$ at saturation; in the ReLU network, which cannot saturate, the activations and gradients explode by a constant factor per layer.
- **Xavier** keeps the $\tanh$ activations at a stable scale across all ten layers; for ReLU it loses a factor $\sqrt{2}$ per layer, exactly the halving of the second moment we computed.
- **He** keeps the ReLU network stable across depth, and is somewhat too large for $\tanh$.

Histograms make the saturation problem of the too-large $\tanh$ network explicit.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(11, 6.5))
for row, name in enumerate(["small (std 0.01)", "large (std 0.2)", "Xavier"]):
    _, _, acts = layer_statistics(inits[name], torch.tanh)
    for col, layer in enumerate([0, 4, 9]):
        axes[row, col].hist(acts[layer].ravel(), bins=60, range=(-1, 1), color="C0")
        axes[row, col].set_title(f"tanh, {name}, layer {layer + 1}", fontsize=9)
        axes[row, col].set_yticks([])
plt.suptitle("Distribution of tanh activations: too small collapses to 0, too large saturates at ±1, Xavier stays spread out")
plt.tight_layout(); plt.show()

The histograms confirm the readings of the curves: with tiny weights every activation collapses onto zero within a few layers, with large weights the activations pile up at $\pm 1$, where the derivative of $\tanh$ is essentially zero and no gradient can pass, and with Xavier initialization the distribution stays spread over $(-1, 1)$ at every depth.

### 4.3 Vanishing and exploding gradients in a deep sigmoid network

Before 2010, deep networks used sigmoid activations and small random weights, and they were nearly impossible to train beyond a few layers. The variance analysis explains why. The sigmoid's derivative is at most $\sigma'(0) = 1/4$, so even with perfectly scaled weights each layer multiplies the backpropagated gradient by at most $1/4$ (and typically less); after $L$ layers the gradient reaching the first layer is scaled by $(1/4)^L$ or worse. This is the **vanishing gradient problem**. Its mirror image, the **exploding gradient problem**, occurs with weights that are too large (or, as we will see in Notebook 08 · Sequence Models: RNNs, LSTMs and GRUs, in recurrent networks where the same matrix is applied many times). The next cell measures the gradient norm per layer at initialization in 20-layer networks.

In [ ]:
DEPTH_DEEP = 20
plt.figure(figsize=(8, 4.5))
for name, act, init in [("sigmoid, Xavier init", torch.sigmoid, inits["Xavier"]),
                        ("tanh, Xavier init", torch.tanh, inits["Xavier"]),
                        ("ReLU, He init", F.relu, inits["He"])]:
    _, g_norm, _ = layer_statistics(init, act, depth=DEPTH_DEEP)
    plt.semilogy(range(1, DEPTH_DEEP + 1), np.maximum(g_norm, 1e-30), "o-", ms=3, label=name)
plt.xlabel("layer (1 = closest to the input)"); plt.ylabel("||dL/dW|| (log)")
plt.title(f"Gradient norm per layer at initialization, {DEPTH_DEEP}-layer networks")
plt.legend(); plt.tight_layout(); plt.show()

The sigmoid network loses roughly an order of magnitude of gradient every one to two layers: the first layers receive gradients many orders of magnitude smaller than the last and would need an absurdly large learning rate to move, while the same learning rate would blow up the last layers. Xavier-initialized $\tanh$ does better because $\tanh'(0) = 1$, but still decays; He-initialized ReLU is flat across all twenty layers. The historical remedies are exactly the ingredients of modern deep learning: ReLU-family activations, principled initialization, normalization layers (Section 5) and skip connections (Notebook 06).

### 4.4 Experiment: does the initialization change what a deep network learns?

Statistics at initialization are one thing; training is what matters. We train a 10-layer ReLU MLP with three initializations, momentum SGD, and the same budget.

In [ ]:
def init_normal(std):
    """Return a function for model.apply(): N(0, std) weights and zero biases for every nn.Linear."""
    def f(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=std)
            nn.init.zeros_(m.bias)
    return f


def init_he(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
        nn.init.zeros_(m.bias)


def init_xavier(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_normal_(m.weight)
        nn.init.zeros_(m.bias)


DEEP = (256,) * 9                       # nine hidden layers -> ten weight matrices
init_results = {}
for name, init in [("small (std 0.01)", init_normal(0.01)), ("PyTorch default", None), ("He (Kaiming)", init_he)]:
    model = make_mlp(hidden=DEEP, init=init)
    opt = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9)
    h = train(model, opt, EPOCHS, verbose=False)
    init_results[name] = h
    print(f"{name:18s} | final train loss {h['train_loss'][-1]:.4f} | val acc {h['val_acc'][-1]:.3f}")

plt.figure(figsize=(8, 4.5))
for name, h in init_results.items():
    plt.plot(smooth(h["step_loss"]), label=name)
plt.axhline(math.log(10), color="gray", ls=":", label="ln 10 (uniform guess)")
plt.xlabel("optimizer step"); plt.ylabel("training loss (smoothed)")
plt.title("10-layer ReLU MLP: training loss for three initializations")
plt.legend(); plt.tight_layout(); plt.show()

The He-initialized network starts learning immediately. The network with tiny weights sits on the plateau at $\ln 10$: its output is essentially the bias of the last layer, and the gradients reaching the other layers are too small to change anything (in a long run such networks sometimes escape the plateau abruptly after many epochs, when random drift finally amplifies the signal, but that is luck, not a strategy). PyTorch's default initialization lies in between: with variance $1/(3 n_{\text{in}})$ per layer the signal shrinks by a factor of about $2.4$ per ReLU layer, which a 10-layer network survives only barely, so training typically starts late and slowly. The lesson generalizes: in a plain deep network the initialization decides whether the first gradient steps carry any information at all.

> **Key idea.** Initialize so that the variance of activations and of gradients stays roughly constant across layers: $\sigma_W^2 = 2/(n_{\text{in}}+n_{\text{out}})$ for tanh-like activations, $2/n_{\text{in}}$ for ReLU. Normalization layers, which we build next, make training far less sensitive to this choice.

## 5. Normalization

### 5.1 Batch Normalization: idea and equations

Section 4 showed how much effort it takes to keep the scale of activations under control by choosing the initial weights, and that control is only guaranteed *at initialization*: as soon as training changes the weights, the distributions of the activations drift. **Batch Normalization** (Ioffe and Szegedy, 2015) takes a more direct approach. Instead of hoping the activations have a good scale, it *forces* them to: for every feature it subtracts the mean and divides by the standard deviation, computed over the current mini-batch, and then lets the network choose its own scale and offset through two learnable parameters.

For a mini-batch $\{x_1, \dots, x_B\}$ of a single feature (in a fully-connected layer, each of the $d$ output units is a feature; the computation below is applied to each independently):

$$
\mu_B = \frac{1}{B}\sum_{i=1}^{B} x_i, \qquad
\sigma_B^2 = \frac{1}{B}\sum_{i=1}^{B} (x_i - \mu_B)^2, \qquad
\hat x_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}, \qquad
y_i = \gamma\, \hat x_i + \beta .
$$

The constant $\epsilon$ (default $10^{-5}$) prevents division by zero, and $\gamma$ (initialized to $1$) and $\beta$ (initialized to $0$) are learned like any other parameters; with $\gamma = \sqrt{\sigma_B^2+\epsilon}$ and $\beta = \mu_B$ the layer could learn to undo the normalization entirely, so it does not restrict what the network can represent. BatchNorm is usually placed between a linear (or convolutional) layer and its activation function; since it subtracts the mean, the bias of the preceding linear layer is redundant and can be dropped.

**Why does it help?** The original paper argued that it reduces *internal covariate shift*, the drift in the distribution of each layer's inputs during training. Later work (Santurkar et al., 2018) found that the more important effect is on the *optimization landscape*: with the activations normalized, the loss is smoother and its gradients are more predictable, which allows much larger learning rates and makes training far less sensitive to initialization. A third effect is **scale invariance**: because the output of the normalization does not change when the weights of the preceding layer are multiplied by a constant, the effective learning rate of those weights adapts automatically to their norm. And a fourth effect is regularization: the statistics of a random mini-batch are noisy, so each example is normalized slightly differently at every step, which acts like a small amount of injected noise.

**Training versus evaluation.** The batch statistics are only available during training, and at test time we want the prediction for an example to depend on that example alone. BatchNorm therefore keeps **running averages** of the mean and variance, updated at every training step with an exponential moving average of momentum $\alpha$ (PyTorch default $0.1$),

$$
\mu_{\text{run}} \leftarrow (1-\alpha)\,\mu_{\text{run}} + \alpha\, \mu_B, \qquad
\sigma^2_{\text{run}} \leftarrow (1-\alpha)\,\sigma^2_{\text{run}} + \alpha\, \frac{B}{B-1}\sigma_B^2 ,
$$

and uses them instead of the batch statistics in evaluation mode. (PyTorch normalizes with the biased batch variance but stores the unbiased estimate, hence the $B/(B-1)$.) The running statistics are **buffers**, not parameters: they are part of the model's `state_dict` but receive no gradient. Note that the gradient of the loss flows through $\mu_B$ and $\sigma_B$ as well as through $x_i$, since they are functions of the batch; autograd handles this automatically, and it matters (a version that treats the statistics as constants trains worse).

### 5.2 BatchNorm from scratch, checked against `nn.BatchNorm1d`

The class below implements the equations of Section 5.1 for `(batch, features)` inputs: batch statistics and a running-average update in training mode, the stored statistics in evaluation mode, and a learnable scale and shift. The check that follows pushes the same random batches through our layer and through `nn.BatchNorm1d` in both modes and compares the outputs and the running statistics.

In [ ]:
class BatchNorm1dScratch(nn.Module):
    """Batch Normalization for (batch, features) inputs, written out in full."""
    def __init__(self, num_features, eps=1e-5, momentum=0.1):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(num_features))      # learnable scale
        self.beta = nn.Parameter(torch.zeros(num_features))      # learnable shift
        self.register_buffer("running_mean", torch.zeros(num_features))
        self.register_buffer("running_var", torch.ones(num_features))
        self.eps, self.momentum = eps, momentum

    def forward(self, x):
        if self.training:
            mean = x.mean(dim=0)
            var = x.var(dim=0, unbiased=False)                   # biased variance normalizes the batch
            with torch.no_grad():                                # running statistics get no gradient
                n = x.shape[0]
                self.running_mean.mul_(1 - self.momentum).add_(self.momentum * mean)
                self.running_var.mul_(1 - self.momentum).add_(self.momentum * var * n / max(n - 1, 1))
        else:
            mean, var = self.running_mean, self.running_var        # evaluation: fixed statistics
        x_hat = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * x_hat + self.beta


torch.manual_seed(SEED)
bn_ours, bn_torch = BatchNorm1dScratch(64).to(device), nn.BatchNorm1d(64).to(device)
for _ in range(5):                                       # five training-mode forward passes
    xb = torch.randn(32, 64, device=device) * 3.0 + 1.0
    diff_train = (bn_ours(xb) - bn_torch(xb)).abs().max().item()
print(f"train mode: max |ours - torch| = {diff_train:.1e}")
print(f"running mean gap {(bn_ours.running_mean - bn_torch.running_mean).abs().max().item():.1e}, "
      f"running var gap {(bn_ours.running_var - bn_torch.running_var).abs().max().item():.1e}")
bn_ours.eval(); bn_torch.eval()
xb = torch.randn(32, 64, device=device) * 3.0 + 1.0
out_ours, out_torch = bn_ours(xb), bn_torch(xb)
print(f"eval mode:  max |ours - torch| = {(out_ours - out_torch).abs().max().item():.1e}")
print(f"in train mode the output has per-feature mean ~0 and std ~1: "
      f"mean {bn_torch.train()(xb).mean().item():.3f}, std {bn_torch(xb).std(dim=0).mean().item():.3f}")

Our layer reproduces PyTorch's to floating-point precision in both modes, including the running statistics. Keep the two-mode behaviour in mind: **`model.train()` and `model.eval()` change what BatchNorm computes**, and forgetting to switch is one of the most common bugs in PyTorch code (Exercise 3 shows what goes wrong). For images, `nn.BatchNorm2d` normalizes each channel over the batch *and* the spatial positions; the equations are identical with the mean taken over $(B, H, W)$.

### 5.3 Experiment: trainability of deep networks and the allowable learning rate

We return to the 10-layer ReLU MLP and compare three variants, plain, with BatchNorm after every hidden linear layer, and with LayerNorm (Section 5.5), each with plain SGD at a moderate learning rate ($0.05$) and a large one ($0.5$). All use He initialization.

In [ ]:
norm_results = {}
for norm in (None, "batch", "layer"):
    for lr in (0.05, 0.5):
        name = f"{norm or 'plain':6s} lr={lr}"
        model = make_mlp(hidden=DEEP, norm=norm, init=init_he)
        opt = torch.optim.SGD(model.parameters(), lr=lr)
        h = train(model, opt, EPOCHS, verbose=False)
        norm_results[name] = h
        print(f"{name:14s} | final train loss {h['train_loss'][-1]:8.4f} | val acc {h['val_acc'][-1]:.3f}")

plt.figure(figsize=(8.5, 4.5))
for name, h in norm_results.items():
    plt.plot(smooth(h["step_loss"]), lw=1, ls="--" if "0.5" in name else "-", label=name)
plt.yscale("log"); plt.xlabel("optimizer step"); plt.ylabel("training loss (smoothed, log)")
plt.title("10-layer ReLU MLP: plain vs BatchNorm vs LayerNorm at two learning rates")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

At the moderate learning rate all three variants train, and the normalized networks usually reach a lower loss sooner. At the large learning rate the plain network typically fails, either diverging to `nan` (the curve disappears from the log plot) or getting stuck at a high loss with many dead units, whereas the BatchNorm network trains comfortably, often better than at the small learning rate. This robustness to the learning rate, together with the insensitivity to initialization, is the practical reason normalization layers are everywhere: they turn the hyperparameter search from a knife-edge into a plateau. (If in your run the plain network survived the large learning rate, increase it further, to $1$ or $2$, and compare again.)

### 5.4 The batch-size dependence of BatchNorm

BatchNorm's Achilles heel is its dependence on the batch. The statistics of a batch of $B$ examples estimate the population statistics with an error of order $1/\sqrt{B}$; for $B = 128$ that is fine, but for $B = 2$ or $4$ (common with large images or 3-D data) each example is normalized against a couple of random neighbours, the training signal becomes very noisy, and the running averages used at test time are poor estimates. The next cell shows the effect from the evaluation side: we take a trained BatchNorm network, switch it to training mode so that it normalizes with the statistics of whatever batch it is given, and measure the validation accuracy as a function of the evaluation batch size. In eval mode the accuracy is independent of the batch size, as it should be. The helper works on a copy of the model: a forward pass in training mode also *updates* the running statistics, and we want `bn_model` intact for Exercise 3.

In [ ]:
bn_model = make_mlp(hidden=DEEP, norm="batch", init=init_he)
opt = torch.optim.SGD(bn_model.parameters(), lr=0.1)
_ = train(bn_model, opt, EPOCHS, verbose=False)
_, acc_eval = evaluate(bn_model, X_val, y_val)
print(f"eval mode (running statistics): val acc {acc_eval:.3f} for any batch size")

def accuracy_with_batch_stats(model, X, y, batch_size):
    """Accuracy when BatchNorm uses the statistics of each evaluation batch (train mode, no gradient)."""
    model = copy.deepcopy(model)          # train-mode passes update the running statistics: keep the original intact
    model.train()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(X) - batch_size + 1, batch_size):
            correct += (model(X[i:i + batch_size]).argmax(dim=1) == y[i:i + batch_size]).sum().item()
    return correct / (len(X) // batch_size * batch_size)

for bs in (2, 4, 8, 32, 128, 1000):
    print(f"train mode (batch statistics), batch size {bs:4d}: val acc {accuracy_with_batch_stats(bn_model, X_val, y_val, bs):.3f}")

With batch statistics from large batches the accuracy matches the eval-mode number, but as the batch shrinks the accuracy collapses: with two examples per batch, the "mean" subtracted from each example is half the example itself. This is the same mechanism that degrades *training* with tiny batches, and it motivated normalization layers that do not look across the batch at all.

**BatchNorm and dropout.** The two do not combine well when dropout precedes BatchNorm: dropout changes the variance of its output between training (where units are randomly zeroed and the rest rescaled) and evaluation (where nothing is zeroed), so the running variance that BatchNorm accumulated during training is wrong at test time (Li et al., 2019, called this *variance shift*). Modern convolutional networks use BatchNorm without dropout; when both are used, dropout goes after all BatchNorm layers, typically only before the final classifier.

### 5.5 LayerNorm and GroupNorm

**Layer Normalization** (Ba, Kiros and Hinton, 2016) normalizes each example over its *features* instead of each feature over the *batch*: for an input vector $x \in \mathbb{R}^d$,

$$
\mu = \frac{1}{d}\sum_{j=1}^{d} x_j, \qquad \sigma^2 = \frac{1}{d}\sum_{j=1}^{d}(x_j - \mu)^2, \qquad
y = \gamma \odot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta ,
$$

with a learnable scale and shift per feature. The statistics involve a single example, so the layer computes exactly the same function in training and evaluation, needs no running averages, works with a batch size of one, and is indifferent to sequence lengths that vary across a batch. That is why Transformers (Notebook 09 · Attention and Transformers) and recurrent networks use LayerNorm: in a language model every token position is normalized independently across its embedding dimension. The price is that LayerNorm assumes all features of an example should share a scale, which is natural for embedding vectors but less so for the channels of a convolutional feature map.

**Group Normalization** (Wu and He, 2018) is the compromise for convolutional networks: the channels are split into groups (32 by default) and each group is normalized over its channels and spatial positions for a single example. It matches BatchNorm at normal batch sizes and does not degrade at tiny ones; **Instance Normalization** (one group per channel) is the special case used in style transfer. All are available as `nn.LayerNorm`, `nn.GroupNorm` and `nn.InstanceNorm2d`. A quick check that `nn.LayerNorm` is the formula above:

In [ ]:
ln = nn.LayerNorm(64).to(device)
xb = torch.randn(32, 64, device=device) * 3.0 + 1.0
manual = (xb - xb.mean(dim=-1, keepdim=True)) / torch.sqrt(xb.var(dim=-1, unbiased=False, keepdim=True) + ln.eps)
print(f"nn.LayerNorm vs manual formula: max gap {(ln(xb) - manual).abs().max().item():.1e}")
print(f"LayerNorm on a single example works: output shape {tuple(ln(xb[:1]).shape)}; "
      f"BatchNorm1d on a single example in train mode raises an error (see Exercise 3)")

> **Key idea.** Normalization layers fix the scale of activations by construction rather than by careful initialization. BatchNorm normalizes each feature across the batch (needs batch statistics, behaves differently in train and eval mode, dislikes tiny batches); LayerNorm normalizes each example across its features (no batch dependence, the choice for sequence models).

## 6. Regularization

### 6.1 Overfitting, seen in learning curves

A model **overfits** when it learns properties of the training set that do not carry over to new data: the training loss keeps falling while the loss on held-out data stops falling and starts to rise. A network with hundreds of thousands of parameters can memorize a few thousand training images perfectly, so with a small training set overfitting is guaranteed unless we do something about it. **Regularization** is the collective name for everything we do to reduce the gap between training and validation performance, usually by accepting a slightly higher training loss in exchange for a lower validation loss.

To make the phenomenon visible we shrink the training set to 2,000 examples, widen the MLP to two hidden layers of 512 units, and train with Adam for many epochs while tracking both losses.

In [ ]:
N_SMALL = 2_000
X_small, y_small = X_train[:N_SMALL], y_train[:N_SMALL]
WIDE = (512, 512)
OVERFIT_EPOCHS = 30 if FAST_RUN else 60

model = make_mlp(hidden=WIDE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
hist_over = train(model, opt, OVERFIT_EPOCHS, X=X_small, y=y_small, verbose=False)
best_epoch = int(np.argmin(hist_over["val_loss"])) + 1
print(f"final: train loss {hist_over['train_loss'][-1]:.4f}, val loss {hist_over['val_loss'][-1]:.4f}, "
      f"train acc {hist_over['train_acc'][-1]:.3f}, val acc {hist_over['val_acc'][-1]:.3f}")
print(f"best validation loss {min(hist_over['val_loss']):.4f} at epoch {best_epoch} "
      f"(val acc there: {hist_over['val_acc'][best_epoch - 1]:.3f})")

epochs_x = range(1, OVERFIT_EPOCHS + 1)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(epochs_x, hist_over["train_loss"], label="train"); ax1.plot(epochs_x, hist_over["val_loss"], label="validation")
ax1.axvline(best_epoch, color="gray", ls=":", label=f"best val loss (epoch {best_epoch})")
ax1.set_xlabel("epoch"); ax1.set_ylabel("cross-entropy"); ax1.set_title("Loss: the classic overfitting picture"); ax1.legend()
ax2.plot(epochs_x, hist_over["train_acc"], label="train"); ax2.plot(epochs_x, hist_over["val_acc"], label="validation")
ax2.set_xlabel("epoch"); ax2.set_ylabel("accuracy"); ax2.set_title("Accuracy"); ax2.legend()
plt.tight_layout(); plt.show()

The training loss heads towards zero and the training accuracy towards 100%: the network is memorizing. The validation loss reaches its minimum after a handful of epochs and then climbs steadily, even though the validation *accuracy* stays roughly flat: the network becomes increasingly, and wrongly, confident on the examples it gets wrong. The vertical line marks where **early stopping** would have halted training (Section 6.4). Everything that follows is a way of moving the validation curve down.

### 6.2 L2 regularization and weight decay

The oldest regularizer adds a penalty on the size of the weights to the loss:

$$
L_{\text{reg}}(\theta) \;=\; L(\theta) + \frac{\lambda}{2}\,\|\theta\|_2^2 , \qquad
\nabla L_{\text{reg}} \;=\; \nabla L + \lambda\theta ,
$$

so that every gradient step also pulls the weights towards zero: $\theta \leftarrow (1 - \eta\lambda)\theta - \eta\nabla L$, the **weight decay** of Section 2.6. Why should small weights generalize better? Three complementary views:

- *Smoothness.* The output of a network with small weights changes slowly with its input (its Lipschitz constant is bounded by a product of weight norms), so it cannot carve out the sharp decision regions needed to memorize individual examples.
- *Bayesian.* Minimizing $L + \frac{\lambda}{2}\|\theta\|^2$ is **maximum a posteriori** estimation with a Gaussian prior $\theta \sim \mathcal{N}(0, \lambda^{-1} I)$ on the weights (take the negative log of prior times likelihood; see Notebook 01 · Mathematical Foundations for the likelihood side): we are declaring that we believe weights are small unless the data insist otherwise.
- *Effective capacity.* For a linear model, L2 shrinks the solution along directions of low data variance, reducing the number of effective degrees of freedom.

Typical values are $\lambda = 5\cdot 10^{-4}$ for SGD on convolutional networks and $\lambda \in [0.01, 0.1]$ for AdamW (decoupled). Biases and normalization parameters are usually excluded. The **L1** penalty $\lambda\|\theta\|_1$ instead drives many weights to exactly zero (sparsity); it is rarely used for deep networks except for pruning (Notebook 15 · Deployment and Model Optimization).

### 6.3 Dropout from scratch

**Dropout** (Srivastava et al., 2014) randomly sets each hidden unit to zero with probability $p$ at every training step, independently per example and per step. A unit cannot rely on any specific other unit being present, so the network is pushed towards redundant, distributed representations, and the whole procedure can be read as training an exponentially large ensemble of thinned sub-networks that share weights.

At test time we want a deterministic prediction, and the naive approach of using all units would change the scale of the activations: with a fraction $1-p$ of the units active during training, the expected input to the next layer at test time would be larger by $1/(1-p)$. **Inverted dropout**, the form used by every modern framework, fixes the scale during training instead: for a mask $m_j \sim \text{Bernoulli}(1-p)$,

$$
\tilde h_j \;=\; \frac{m_j\, h_j}{1-p}, \qquad \mathbb{E}[\tilde h_j] \;=\; \frac{(1-p)\, h_j}{1-p} \;=\; h_j ,
$$

so the expected activation is unchanged and the layer is simply the identity at evaluation time. Typical values: $p = 0.5$ for large fully-connected layers, $0.1$–$0.3$ in Transformers, close to $0$ in modern convolutional networks (which rely on BatchNorm and augmentation). Dropout increases the training loss and slows convergence, so it is a tool for the overfitting regime, not a default.

In [ ]:
class DropoutScratch(nn.Module):
    """Inverted dropout: zero each unit with probability p in training, scale the rest by 1/(1-p); identity in eval."""
    def __init__(self, p=0.5):
        super().__init__()
        self.p = p

    def forward(self, x):
        if not self.training or self.p == 0.0:
            return x
        mask = (torch.rand_like(x) >= self.p).to(x.dtype)
        return x * mask / (1 - self.p)


torch.manual_seed(SEED)
h = torch.full((200_000,), 2.0, device=device)
drop_ours, drop_torch = DropoutScratch(0.5), nn.Dropout(0.5)
out = drop_ours(h)
print(f"train mode: fraction zeroed {(out == 0).float().mean().item():.3f}, surviving values equal {out.max().item():.1f}, "
      f"mean {out.mean().item():.3f} (input mean 2.0)")
print(f"nn.Dropout for comparison: mean {drop_torch(h).mean().item():.3f}")
drop_ours.eval()
print(f"eval mode: output identical to input: {torch.equal(drop_ours(h), h)}")

Half of the units are zeroed, the survivors are doubled, the mean is preserved, and in eval mode the layer disappears. `nn.Dropout` behaves identically. Our `make_mlp` factory inserts `nn.Dropout` after each hidden activation when `dropout > 0`, which we use in the comparison of Section 6.9.

### 6.4 Early stopping with a patience loop

Early stopping is the regularizer you get for free by watching the validation loss: train as long as it improves, keep a copy of the best model, and stop after `patience` epochs without improvement. It is cheap, it needs no hyperparameter beyond the patience, and it combines with everything else. In Notebook 05 · The PyTorch Workflow and Best Practices it becomes part of a reusable `fit` function; here is the minimal version, built on our `train` by running one epoch at a time. Note the `copy.deepcopy` of the `state_dict`: without it, the "best" state would be a reference to tensors that keep changing.

In [ ]:
def train_with_early_stopping(model, opt, max_epochs, patience, X, y, min_delta=1e-4, verbose=True):
    """Train one epoch at a time; stop after `patience` epochs without a validation-loss improvement.

    Restores the best weights before returning. Returns (history, best_epoch).
    """
    best_loss, best_state, best_epoch, bad_epochs = float("inf"), None, 0, 0
    hist = {"train_loss": [], "val_loss": [], "val_acc": []}
    for epoch in range(1, max_epochs + 1):
        h = train(model, opt, 1, X=X, y=y, verbose=False, seed=SEED + epoch)   # a new shuffle each epoch
        for k in hist:
            hist[k].append(h[k][0])
        if h["val_loss"][0] < best_loss - min_delta:
            best_loss, best_epoch, bad_epochs = h["val_loss"][0], epoch, 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                if verbose:
                    print(f"early stopping at epoch {epoch}: no improvement for {patience} epochs")
                break
    model.load_state_dict(best_state)
    return hist, best_epoch


model = make_mlp(hidden=WIDE)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
hist_es, best_epoch_es = train_with_early_stopping(model, opt, OVERFIT_EPOCHS, patience=5, X=X_small, y=y_small)
print(f"trained for {len(hist_es['val_loss'])} epochs, restored epoch {best_epoch_es}: "
      f"val loss {hist_es['val_loss'][best_epoch_es - 1]:.4f}, val acc {hist_es['val_acc'][best_epoch_es - 1]:.3f}")

Compared with the run of Section 6.1, we stopped after a fraction of the epochs and kept the weights with the lowest validation loss. The patience should be large enough to ride out the ordinary epoch-to-epoch noise of the validation loss (5–10 epochs is common) and the metric you stop on should be the one you care about. One caveat: the validation set is now being used to choose the stopping point, which is a (mild) form of fitting to it, so the final estimate of generalization must come from the untouched test set.

### 6.5 Data augmentation as regularization (a preview)

The most effective regularizer for perceptual data is more data, and **data augmentation** manufactures it: every time an image is used for training it is randomly transformed in a way that preserves its label (shifted, flipped, rotated, cropped, recoloured). The network then cannot memorize individual pixels, because it never sees the same pixels twice, and it is pushed to learn features that are invariant to the transformations. Notebook 06 · Convolutional Neural Networks uses `torchvision.transforms.v2` for this; here we write a two-line version directly on our flattened tensors: a random horizontal flip per example and a random shift of up to two pixels per batch (padding with the standardized value of a black pixel). Horizontal flips are label-preserving for clothes, which is not true for all data: a flipped "6" is not a "6".

In [ ]:
BLACK = float((0.0 - MEAN) / STD)               # standardized value of a black pixel


def augment_shift_flip(xb, max_shift=2):
    """Random horizontal flip (per example) and random translation (per batch) on flattened 28x28 images."""
    img = xb.reshape(-1, 1, 28, 28)
    flip = torch.rand(img.shape[0], 1, 1, 1, device=img.device) < 0.5
    img = torch.where(flip, img.flip(-1), img)
    padded = F.pad(img, (max_shift,) * 4, value=BLACK)
    dx, dy = torch.randint(0, 2 * max_shift + 1, (2,)).tolist()
    img = padded[:, :, dy:dy + 28, dx:dx + 28]
    return img.reshape(-1, 784)


torch.manual_seed(SEED)
orig = X_train[:6]
fig, axes = plt.subplots(3, 6, figsize=(9, 4.5))
for row, batch in enumerate([orig, augment_shift_flip(orig), augment_shift_flip(orig)]):
    imgs = (batch.reshape(-1, 28, 28).cpu() * STD + MEAN).cpu().numpy()      # undo the standardization for display
    for col in range(6):
        axes[row, col].imshow(imgs[col], cmap="gray", vmin=0, vmax=1); axes[row, col].axis("off")
    axes[row, 0].set_title("original" if row == 0 else f"augmented #{row}", fontsize=9, loc="left")
plt.tight_layout(); plt.show()

Each row of the figure shows the same six images; the second and third rows are two independent draws of the flip and the shift, so the network never sees exactly the same pixels twice. Because the augmentation is applied on the fly inside `train` (through the `augment` hook), it costs no memory and no preprocessing; we use it in the comparison of Section 6.9.

### 6.6 Label smoothing

Cross-entropy with one-hot targets asks the network for infinite confidence: the loss is minimized only when the logit of the correct class is infinitely larger than the others, which drives weights to grow and calibrates the model poorly. **Label smoothing** (Szegedy et al., 2016) replaces the one-hot target $y$ by a mixture with the uniform distribution, $\tilde y = (1-\alpha)\, y + \alpha / K$ for $K$ classes, so that the loss becomes

$$
\ell_{\text{LS}} \;=\; -(1-\alpha)\log p_{y} \;-\; \frac{\alpha}{K}\sum_{k=1}^{K}\log p_k ,
$$

which is minimized by predicting probability $1 - \alpha + \alpha/K$ for the correct class and $\alpha/K$ for each other class. Typical $\alpha = 0.1$. In PyTorch it is a keyword: `F.cross_entropy(logits, y, label_smoothing=0.1)`. It regularizes slightly, improves calibration, and is standard in image classification and machine translation recipes.

### 6.7 Gradient clipping

Clipping does not reduce overfitting; it protects against the rare mini-batch whose gradient is enormous (a mislabelled example, an exploding recurrent state, an unlucky moment with a large learning rate) and would otherwise throw the parameters into a bad region. **Clipping by norm** rescales the whole gradient vector so that its norm is at most $c$,

$$
g \;\leftarrow\; g \cdot \min\!\left(1, \frac{c}{\|g\|_2}\right),
$$

which preserves the direction; **clipping by value** clamps each component to $[-c, c]$, which does not. Norm clipping with $c = 1$ is standard for Transformers and recurrent networks; our `train` function applies `torch.nn.utils.clip_grad_norm_` when `clip_norm` is given, and Notebook 08 · Sequence Models: RNNs, LSTMs and GRUs shows why recurrent networks need it. The function returns the norm *before* clipping, which is worth logging: a sudden growth of the gradient norm is an early warning of instability.

In [ ]:
model = make_mlp()
loss = F.cross_entropy(model(X_train[:BATCH]) * 50, y_train[:BATCH])       # scaled logits: a huge gradient
model.zero_grad(); loss.backward()
norm_before = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
norm_after = math.sqrt(sum(p.grad.norm().item() ** 2 for p in model.parameters()))
print(f"gradient norm before clipping {norm_before.item():.2f}, after clipping {norm_after:.2f}")

The scaled logits produce a gradient whose norm is in the hundreds; after `clip_grad_norm_` it is exactly $1$ (up to float32 round-off in the last digits), and the direction is unchanged because every component was multiplied by the same factor $c/\|g\|_2$.

### 6.8 The bias–variance view

For a regression problem with noisy targets $y = f(x) + \varepsilon$, the expected squared error of a learned predictor $\hat f$ decomposes (over random training sets) as

$$
\mathbb{E}\big[(y - \hat f(x))^2\big] \;=\; \underbrace{\big(f(x) - \mathbb{E}[\hat f(x)]\big)^2}_{\text{bias}^2}
\;+\; \underbrace{\mathbb{E}\big[(\hat f(x) - \mathbb{E}[\hat f(x)])^2\big]}_{\text{variance}}
\;+\; \underbrace{\sigma_\varepsilon^2}_{\text{irreducible noise}} .
$$

A model that is too simple has high bias (it cannot represent $f$); a model that is too flexible has high variance (it fits the noise of the particular training set it saw). Every regularizer in this section trades a little bias for a lot of variance: weight decay and dropout constrain the function class, early stopping limits how far the fit can go, augmentation and label smoothing change the target distribution. The classical picture predicts a U-shaped test error as capacity grows; deep networks complicate it (very large, well-regularized networks often generalize *better* than medium ones, the "double descent" phenomenon), but as a guide to what to try when the validation loss rises, the bias–variance view remains the right first instinct: high variance calls for regularization or more data, high bias for a bigger model or longer training.

### 6.9 Comparison experiment

We now put the regularizers side by side on the small-data problem of Section 6.1: the same wide MLP and the same budget, trained with AdamW at learning rate $10^{-3}$ and, in turn, no regularization, decoupled weight decay $0.1$, dropout $0.5$, label smoothing $0.1$, augmentation, and all four together. For each run we report the final training and validation accuracy, the gap between them, the best validation accuracy over the run (the early-stopping result), and the final norm of the weights. Under `FAST_RUN` the budget is 15 epochs; the full run uses 40, where overfitting is stronger and the differences larger.

In [ ]:
REG_EPOCHS = 15 if FAST_RUN else 40
ce_smooth = lambda logits, yb: F.cross_entropy(logits, yb, label_smoothing=0.1)
reg_configs = [
    ("no regularization", dict(), 0.0, dict()),
    ("weight decay 0.1", dict(), 0.1, dict()),
    ("dropout 0.5", dict(dropout=0.5), 0.0, dict()),
    ("label smoothing 0.1", dict(), 0.0, dict(loss_fn=ce_smooth)),
    ("augmentation", dict(), 0.0, dict(augment=augment_shift_flip)),
    ("all combined", dict(dropout=0.5), 0.1, dict(loss_fn=ce_smooth, augment=augment_shift_flip)),
]
print(f"{'regularizer':22s} {'train acc':>9s} {'val acc':>8s} {'gap':>6s} {'best val':>9s} {'||w||':>7s}")
reg_rows = {}
for name, model_kw, wd, train_kw in reg_configs:
    torch.manual_seed(SEED)                                   # same dropout / augmentation randomness per run
    model = make_mlp(hidden=WIDE, **model_kw)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=wd)
    h = train(model, opt, REG_EPOCHS, X=X_small, y=y_small, verbose=False, **train_kw)
    w_norm = math.sqrt(sum((p ** 2).sum().item() for p in model.parameters()))
    reg_rows[name] = h
    print(f"{name:22s} {h['train_acc'][-1]:9.3f} {h['val_acc'][-1]:8.3f} {h['train_acc'][-1] - h['val_acc'][-1]:6.3f} "
          f"{max(h['val_acc']):9.3f} {w_norm:7.1f}")

plt.figure(figsize=(8.5, 4.5))
for name, h in reg_rows.items():
    plt.plot(range(1, REG_EPOCHS + 1), h["val_loss"], label=name)
plt.xlabel("epoch"); plt.ylabel("validation loss"); plt.title("Validation loss under different regularizers (2,000 training examples)")
plt.legend(fontsize=8); plt.tight_layout(); plt.show()

Read the table column by column. The unregularized network has the highest training accuracy and the largest gap. Weight decay keeps the weight norm small (compare the last column) and reduces the gap modestly. Dropout lowers the training accuracy substantially, which is expected since half of the units are missing at every step, and typically buys a lower validation loss late in the run. Label smoothing mostly changes the validation *loss* (the network is less over-confident) with a smaller effect on accuracy. Augmentation is usually the single most effective ingredient on image data, and the combination generalizes best, at the cost of the lowest training accuracy: the regularized network is worse at memorizing and better at predicting. The validation-loss curves show the same story dynamically: the unregularized curve turns upward early, the regularized ones flatten or keep decreasing. Exact numbers vary with the seed, and on a small validation set differences of half a percent are noise; run the cell with a different `SEED` before drawing fine conclusions.

> **Common pitfall.** Regularizing a model that is *underfitting*. If the training accuracy is already low, dropout and weight decay make it worse; first make the model fit the training data (bigger model, better optimizer, longer training), then regularize.

## 7. Diagnosing training

Most of the time spent on deep learning is spent looking at a loss curve and asking "is this normal?". This section collects the patterns you will meet, using runs from earlier in the notebook plus two deliberately bad ones.

### 7.1 Reading loss curves

The six panels below show the characteristic shapes. Two of them are new runs: plain SGD with a learning rate a hundred times too small, and with a learning rate thirty times too large.

In [ ]:
model = make_mlp(); hist_low = train(model, SGD(model.parameters(), lr=0.001), EPOCHS, verbose=False)
model = make_mlp(); hist_high = train(model, SGD(model.parameters(), lr=3.0), EPOCHS, verbose=False)
model_hot = model                                                     # kept for the dead-unit check below

fig, axes = plt.subplots(2, 3, figsize=(13, 7))
panels = [
    (axes[0, 0], "Learning rate too low", hist_low["step_loss"], "loss decreases very slowly; still near ln 10"),
    (axes[0, 1], "Learning rate too high", hist_high["step_loss"], "loss jumps around, spikes, or becomes nan"),
    (axes[0, 2], "Healthy run (Nesterov)", optim_results["Nesterov lr=0.02"]["step_loss"], "fast early drop, then a slow noisy decline"),
    (axes[1, 0], "Bad initialization", init_results["small (std 0.01)"]["step_loss"], "flat plateau at ln 10, nothing happens"),
]
for ax, title, losses, note in panels:
    ax.plot(losses, lw=0.6, alpha=0.5, color="C0"); ax.plot(np.arange(len(smooth(losses))) + 10, smooth(losses), color="C0")
    ax.axhline(math.log(10), color="gray", ls=":")
    ax.set_title(title); ax.set_xlabel("step"); ax.set_ylabel("mini-batch loss"); ax.set_ylim(0, 3)
    ax.text(0.02, 0.05, note, transform=ax.transAxes, fontsize=8)
ax = axes[1, 1]
ax.plot(hist_over["train_loss"], label="train"); ax.plot(hist_over["val_loss"], label="validation")
ax.set_title("Overfitting"); ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend()
ax.text(0.02, 0.05, "train keeps falling, validation turns upward", transform=ax.transAxes, fontsize=8)
ax = axes[1, 2]
ax.plot(bs_results[256]["train_loss"], label="train"); ax.plot(bs_results[256]["val_loss"], label="validation")
ax.set_title("Underfitting / too short"); ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.legend()
ax.text(0.02, 0.05, "both losses still high and falling: train longer or bigger", transform=ax.transAxes, fontsize=8)
plt.tight_layout(); plt.show()

A short field guide to these shapes:

- **Learning rate too low**: the loss decreases, but so slowly that the curve looks flat; the run wastes its budget. Multiply the learning rate by 3–10, or run the range test of Section 3.2.
- **Learning rate too high**: the loss bounces around a high value, spikes, or becomes `nan`; with ReLU networks, a violent early step can also kill a large fraction of the units (Section 7.2), after which the loss settles at a mediocre value and never recovers. Divide the learning rate by 3–10 or add warmup.
- **Healthy**: a fast initial drop followed by a slow, noisy decline whose noise band shrinks if the learning rate is decayed.
- **Bad initialization**: an exactly flat plateau at the loss of a uniform prediction ($\ln K$ for $K$ classes), often followed by a sudden drop much later. Check the initialization (Section 4) or add normalization.
- **Overfitting**: training loss keeps falling while validation loss rises. Regularize (Section 6), add data, or stop early.
- **Underfitting**: both losses are high and still decreasing when the budget ends, or plateau together at a high value. Train longer, increase model capacity, or increase the learning rate; do not regularize.

Two more signatures worth knowing. A loss that decreases and then *abruptly* jumps up and stays there often indicates a learning-rate schedule bug or a numerical blow-up (check gradient norms). A validation loss that is *lower* than the training loss early in training is normal when dropout or augmentation is on (the training loss is measured with the noise switched on, the validation loss without), and it is also common in code that reports the *average* of the mini-batch losses of an epoch, because the model improves during the epoch and that average lags behind the model that is validated at its end. Our `train` function avoids the second effect: it re-evaluates the training loss on the whole training set at the end of every epoch, in eval mode, so `train_loss` and `val_loss` are directly comparable; only `step_loss` holds the noisy in-training values.

### 7.2 Dead ReLUs

A ReLU unit whose pre-activation is negative for every input produces zero output and, because the ReLU gradient is zero there, receives zero gradient: it is **dead** and will stay dead. A few dead units are harmless; a large fraction means the network has lost capacity, typically after an overly large learning rate or a bad initialization pushed biases far negative. The check is simple: run the data through the network and count units that are never active.

In [ ]:
@torch.no_grad()
def dead_unit_fraction(model, X):
    """Fraction of ReLU units that are never active on X, per ReLU layer of an nn.Sequential."""
    model.eval()
    h, fracs = X, []
    for layer in model:
        h = layer(h)
        if isinstance(layer, nn.ReLU):
            fracs.append((h.amax(dim=0) <= 0).float().mean().item())
    return fracs


healthy = make_mlp(); _ = train(healthy, torch.optim.SGD(healthy.parameters(), lr=0.02, momentum=0.9, nesterov=True), EPOCHS, verbose=False)
print(f"healthy run (Nesterov lr=0.02): dead units per layer {[round(f, 3) for f in dead_unit_fraction(healthy, X_val)]}")
print(f"hot run (SGD lr=3.0):           dead units per layer {[round(f, 3) for f in dead_unit_fraction(model_hot, X_val)]}")
print(f"untrained network:              dead units per layer {[round(f, 3) for f in dead_unit_fraction(make_mlp(), X_val)]}")

The healthy network has (almost) no dead units. The network trained with the far too large learning rate is a different story: either its parameters are `nan` (in which case the count is meaningless, and the `nan` loss is the diagnosis), or a substantial fraction of its units never fire. The remedies are a smaller learning rate, warmup, normalization, or a leaky variant of ReLU that never has an exactly zero gradient.

### 7.3 The "overfit one batch" sanity check

Before training anything for hours, check that your model *can* learn: take a single small mini-batch and train on it repeatedly. A correct model with a correct loss should drive the loss on that batch essentially to zero within a few hundred steps, because memorizing 64 examples is trivial for any network with thousands of parameters. If the loss plateaus, something is wrong in the pipeline: a bug in the model, a loss applied to the wrong quantity, mismatched labels, an optimizer that never sees the parameters, gradients that are zeroed too late. We demonstrate the check on a correct model and on one with a classic bug: applying a softmax *before* `F.cross_entropy`, which already contains a softmax. The double softmax squashes the logits into $[0, 1]$, so the predicted probabilities can never exceed $e/(e + 9) \approx 0.23$ and the loss cannot fall below about $1.46$.

In [ ]:
def overfit_one_batch(model, opt, xb, yb, steps, loss_fn=F.cross_entropy):
    model.train()
    losses = []
    for _ in range(steps):
        loss = loss_fn(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    return losses


ONE_BATCH_STEPS = 200 if FAST_RUN else 500
xb, yb = X_train[:64], y_train[:64]
model_ok = make_mlp()
losses_ok = overfit_one_batch(model_ok, torch.optim.Adam(model_ok.parameters(), lr=1e-3), xb, yb, ONE_BATCH_STEPS)
model_bug = make_mlp()
double_softmax_loss = lambda logits, y: F.cross_entropy(F.softmax(logits, dim=1), y)      # the bug
losses_bug = overfit_one_batch(model_bug, torch.optim.Adam(model_bug.parameters(), lr=1e-3), xb, yb,
                               ONE_BATCH_STEPS, loss_fn=double_softmax_loss)
plt.figure()
plt.semilogy(losses_ok, label="correct model: loss goes to ~0")
plt.semilogy(losses_bug, label="softmax before cross_entropy: stuck above 1.46")
plt.axhline(-math.log(math.e / (math.e + 9)), color="gray", ls=":")
plt.xlabel("step"); plt.ylabel("loss on the single batch (log)"); plt.title("Overfit-one-batch sanity check")
plt.legend(); plt.tight_layout(); plt.show()
print(f"final loss: correct {losses_ok[-1]:.4f}, buggy {losses_bug[-1]:.4f}")

The correct model memorizes the batch; the buggy one gets stuck at the theoretical floor, and the check has found in seconds a bug that would otherwise manifest as "the model trains but never gets good". Notebook 05 · The PyTorch Workflow and Best Practices adds more debugging tools (shape assertions, hooks, anomaly detection).

### 7.4 NaN losses and their causes

A `nan` (or `inf`) loss is the most alarming symptom and usually the easiest to explain. The usual suspects, roughly in order of frequency:

1. **Learning rate too high**: parameters grow until an activation or logit overflows float32 ($3.4\cdot 10^{38}$). The loss typically rises for a few steps before becoming `inf`, then `nan`. Reduce the learning rate, add warmup, clip gradients.
2. **Logarithm of zero or division by zero** in a hand-written loss: `torch.log(p)` with $p = 0$, a variance of zero in a hand-written normalization without $\epsilon$, `x / x.sum()` on an all-zero row. Use `F.cross_entropy` on logits (it uses the log-sum-exp trick), `torch.log(p + 1e-8)` or `torch.clamp`, and always add $\epsilon$ under square roots.
3. **Exploding gradients** in deep or recurrent networks: clip by norm and check the initialization.
4. **Float16 overflow** under mixed precision: half precision overflows at $65{,}504$; use bfloat16 or the gradient scaler of `torch.amp` (Notebook 14 · Efficient Training and Scaling).
5. **Bad inputs**: `nan` in the data itself (a missing value in a table, a corrupted image). Check `torch.isfinite(X).all()` once, at loading time.
6. **Invalid operations on the wrong values**: `sqrt` of a negative number from round-off, `acos` of $1.0000001$, `pow` of a negative base with a fractional exponent.

A minimal defence is to check `torch.isfinite(loss)` in the training loop and stop with a clear message (Notebook 05's `fit` function does this), and to log the gradient norm returned by `clip_grad_norm_`. For finding *where* a `nan` first appears, `torch.autograd.set_detect_anomaly(True)` makes the backward pass raise an error at the operation that produced it, at the cost of a large slowdown; use it only for debugging.

### 7.5 A checklist

When a training run misbehaves, work through the list in order; most problems are found in the first half.

1. Does the loss start at the expected value ($\ln K$ for a $K$-class softmax with a small-weight initialization)? If not, the head or the loss is wrong.
2. Can the model overfit one batch? If not, there is a bug, not a hyperparameter problem.
3. Is the model in the right mode (`train()` / `eval()`), and are gradients zeroed every step?
4. Is the data what you think: correct labels, standardized inputs, augmentation that preserves labels, no leakage between train and validation, shuffled training batches?
5. Run a learning-rate range test; then try the learning rate $3\times$ up and $3\times$ down.
6. Look at the gradient norm and at the fraction of dead units; check the initialization of deep plain networks or add normalization.
7. If training is fine but validation is not: regularize, augment, or get more data. If neither is fine: bigger model, longer training, better optimizer.
8. Change one thing at a time and log everything (Notebook 05 sets up TensorBoard for this).

## 8. Hyperparameter search basics

### 8.1 Grid search versus random search

Learning rate, weight decay, width, depth, dropout, batch size, schedule: even a small model has half a dozen hyperparameters, and the loss is expensive to evaluate for each combination. **Grid search** evaluates all combinations of a few values per hyperparameter; **random search** samples each hyperparameter independently from a distribution. Bergstra and Bengio (2012) showed, empirically and by a simple argument, that random search is almost always better for the same budget. The argument: in most problems only one or two hyperparameters matter much (the learning rate, above all). A $3\times 3$ grid over an important and an unimportant hyperparameter tries only *three* distinct values of the important one, wasting two thirds of the budget on repeating them; nine random points try nine distinct values. The figure makes the point.

In [ ]:
rng = np.random.default_rng(SEED)
grid_pts = np.array([(i, j) for i in np.linspace(0.15, 0.85, 3) for j in np.linspace(0.15, 0.85, 3)])
rand_pts = rng.uniform(0.02, 0.98, size=(9, 2))
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, pts, name in [(axes[0], grid_pts, "Grid search"), (axes[1], rand_pts, "Random search")]:
    ax.scatter(pts[:, 0], pts[:, 1], s=50)
    ax.scatter(pts[:, 0], np.full(len(pts), -0.06), marker="|", s=250, color="C3")
    n_distinct = len(np.unique(np.round(pts[:, 0], 3)))
    ax.set_title(f"{name}: 9 trials, {n_distinct} distinct values of the important parameter")
    ax.set_xlim(0, 1); ax.set_ylim(-0.12, 1); ax.set_xlabel("important hyperparameter (projection in red)")
    ax.set_ylabel("unimportant hyperparameter")
plt.tight_layout(); plt.show()

### 8.2 Sampling on the right scale

Which distribution to sample from matters as much as the method. The learning rate and the weight decay live on a **logarithmic scale**: the difference between $10^{-4}$ and $10^{-3}$ is as important as that between $10^{-3}$ and $10^{-2}$, and a uniform sample from $[10^{-4}, 10^{-1}]$ would spend 90% of its draws above $10^{-2}$. The right distribution is **log-uniform**: sample an exponent uniformly, $u \sim \mathcal{U}(a, b)$, and use $10^{u}$. Discrete choices (width, number of layers, activation) are sampled from a list; probabilities like dropout are sampled uniformly on their natural range. It is also good practice to make the search space generous the first time (three or four orders of magnitude for the learning rate) and to narrow it around the best region in a second round.

### 8.3 A small random search on the harness

We search four hyperparameters of the MLP with AdamW: the learning rate (log-uniform on $[10^{-4}, 10^{-1.5}]$), the weight decay (log-uniform on $[10^{-5}, 10^{-1}]$), the width of the first hidden layer (128, 256 or 512; the second layer has half that) and the dropout probability (0, 0.2 or 0.5). Each trial trains for a few epochs and reports the best validation accuracy; this "short-run proxy" is the standard compromise, and the reason why Notebook 14 · Efficient Training and Scaling introduces *pruning* of unpromising trials with Optuna.

In [ ]:
N_TRIALS = 8 if FAST_RUN else 20
SEARCH_EPOCHS = 2 if FAST_RUN else 5
rng = np.random.default_rng(SEED)
trials = []
t0 = time.time()
for trial in range(1, N_TRIALS + 1):
    cfg = dict(lr=float(10 ** rng.uniform(-4, -1.5)),
               weight_decay=float(10 ** rng.uniform(-5, -1)),
               width=int(rng.choice([128, 256, 512])),
               dropout=float(rng.choice([0.0, 0.2, 0.5])))
    model = make_mlp(hidden=(cfg["width"], cfg["width"] // 2), dropout=cfg["dropout"])
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    h = train(model, opt, SEARCH_EPOCHS, verbose=False)
    cfg["val_acc"], cfg["val_loss"] = max(h["val_acc"]), min(h["val_loss"])
    trials.append(cfg)
    print(f"trial {trial:2d}: lr {cfg['lr']:.1e} | wd {cfg['weight_decay']:.1e} | width {cfg['width']:3d} | "
          f"dropout {cfg['dropout']:.1f} -> val acc {cfg['val_acc']:.3f}  ({time.time() - t0:.0f}s)")

ranked = sorted(trials, key=lambda c: -c["val_acc"])
best = ranked[0]
print(f"\nbest trial: lr {best['lr']:.2e}, wd {best['weight_decay']:.1e}, width {best['width']}, "
      f"dropout {best['dropout']}, val acc {best['val_acc']:.3f}")

plt.figure(figsize=(7.5, 4.2))
for d, marker in [(0.0, "o"), (0.2, "s"), (0.5, "^")]:
    sel = [c for c in trials if c["dropout"] == d]
    if sel:
        plt.scatter([c["lr"] for c in sel], [c["val_acc"] for c in sel], marker=marker, s=60, label=f"dropout {d}")
plt.xscale("log"); plt.xlabel("learning rate (log scale)"); plt.ylabel("best validation accuracy")
plt.title("Random search: validation accuracy vs learning rate"); plt.legend()
plt.tight_layout(); plt.show()

Even with a handful of trials the scatter plot shows the typical structure: the learning rate dominates, with a broad good region around $10^{-3}$ and poor results at both ends, while the other hyperparameters produce smaller shifts within that region. A search like this is the moment to look at *marginal* effects (accuracy versus each hyperparameter separately) rather than only at the single best trial, since the best trial's other settings may be irrelevant luck.

Finally, the test set. It has been untouched so far, which is exactly what makes it useful: we retrain the best configuration with the full budget and evaluate it **once** on the test set. This is the only number in the notebook that estimates real generalization; everything else was measured on the validation set that we used to make decisions.

In [ ]:
model = make_mlp(hidden=(best["width"], best["width"] // 2), dropout=best["dropout"])
opt = torch.optim.AdamW(model.parameters(), lr=best["lr"], weight_decay=best["weight_decay"])
sched = CosineAnnealingLR(opt, T_max=T_TOTAL)
h = train(model, opt, EPOCHS, scheduler=sched, verbose=False)
test_loss, test_acc = evaluate(model, X_test, y_test)
print(f"best configuration retrained for {EPOCHS} epochs: val acc {h['val_acc'][-1]:.3f} | "
      f"test loss {test_loss:.4f} | test acc {test_acc:.3f} (10,000 test images)")

With the small training subset of this notebook, an MLP lands around 85–88% test accuracy; Notebook 05 · The PyTorch Workflow and Best Practices trains on the full 60,000 images and reaches about 90% with an MLP and more with a small CNN. More sophisticated search methods, Bayesian optimization with Gaussian processes or the tree-structured Parzen estimator, Hyperband and successive halving, population-based training, all build on the same foundations (log-scale sampling, short-run proxies, one held-out set for decisions and another for the final number); Notebook 14 · Efficient Training and Scaling runs a full Optuna study on this harness.

## Exercises

**Exercise 1 (warm-up).** Implement **cosine annealing with warm restarts** (SGDR, Loshchilov and Hutter, 2017) as a factor function for `LambdaLR`: the learning rate follows a cosine from $\eta_0$ to $0$ over $T_0$ steps, then restarts at $\eta_0$ with a period multiplied by `T_mult`. Plot it for $T_0 = 100$, `T_mult = 2` over 700 steps, verify against `torch.optim.lr_scheduler.CosineAnnealingWarmRestarts`, and train the harness MLP with it (one restart period per epoch, doubling).

**Solution to Exercise 1**

The factor function needs to know in which cycle the current step falls; a short loop subtracts completed periods. Restarts give the optimizer several chances to escape a poor basin, and the final model is usually taken at the end of a cycle, when the learning rate is small.

In [ ]:
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts


def cosine_with_restarts(T_0, T_mult=2, min_frac=0.0):
    """LambdaLR factor: cosine from 1 to min_frac over T_0 steps, then restart with the period times T_mult."""
    def factor(step):
        T_i, t = T_0, step
        while t >= T_i:
            t -= T_i
            T_i *= T_mult
        return min_frac + (1 - min_frac) * 0.5 * (1 + math.cos(math.pi * t / T_i))
    return factor


ours = schedule_curve(lambda o: LambdaLR(o, cosine_with_restarts(100, 2)), 700)
ref = schedule_curve(lambda o: CosineAnnealingWarmRestarts(o, T_0=100, T_mult=2), 700)
print(f"max difference to CosineAnnealingWarmRestarts: {max(abs(a - b) for a, b in zip(ours, ref)):.1e}")
plt.figure(); plt.plot(ours, label="ours (LambdaLR)"); plt.plot(ref, "--", label="torch CosineAnnealingWarmRestarts")
plt.xlabel("step"); plt.ylabel("learning rate"); plt.title("Cosine annealing with warm restarts (T_0 = 100, T_mult = 2)")
plt.legend(); plt.tight_layout(); plt.show()

model = make_mlp()
opt = torch.optim.SGD(model.parameters(), lr=LR0, momentum=0.9, nesterov=True)
h_sgdr = train(model, opt, EPOCHS, scheduler=LambdaLR(opt, cosine_with_restarts(steps_per_epoch, 2)), verbose=False)
print(f"SGDR on the harness: val acc per epoch {[round(a, 3) for a in h_sgdr['val_acc']]}; "
      f"plain cosine for comparison: {[round(a, 3) for a in sched_results['cosine']['val_acc']]}")

The restart schedule matches PyTorch's to round-off. On the harness the cycle boundaries fall at the end of epochs 1, 3 and 7 (periods of one, two and four epochs); with a ten-epoch budget the run stops in the middle of the fourth cycle, at a learning rate that is still fairly high, which is why the accuracy is listed per epoch rather than only at the end. After each restart the accuracy typically dips for an epoch and recovers as the learning rate anneals; over a budget this short, SGDR and plain cosine end up within noise of each other, and the benefit of restarts (several low-learning-rate snapshots to choose from, or to average) shows only in longer runs.

**Exercise 2.** Implement **Nadam** (Dozat, 2016), Adam with Nesterov momentum, as a subclass of our `Adam`. In its simplest form the update replaces $\hat m_t$ in the Adam step by a look-ahead combination of the bias-corrected momentum and the current gradient:

$$
\theta_t = \theta_{t-1} - \eta\,\frac{\beta_1 \hat m_t + \dfrac{1-\beta_1}{1-\beta_1^{\,t}}\, g_t}{\sqrt{\hat s_t} + \epsilon}.
$$

Compare it with Adam on the harness at the same learning rate. (PyTorch's `torch.optim.NAdam` additionally schedules $\beta_1$ over time, so it will not match this version exactly.)

**Solution to Exercise 2**

Only the numerator changes, so we override `adam_step`. The extra term $(1-\beta_1) g_t / (1-\beta_1^t)$ is the "look ahead" of Section 2.2 transplanted into Adam: the current gradient gets a slightly larger weight than it has in $\hat m_t$.

In [ ]:
class Nadam(Adam):
    """Adam with Nesterov momentum (Dozat, 2016), without the momentum-decay schedule."""
    def adam_step(self, p, g, state):
        m = state.setdefault("m", torch.zeros_like(p))
        s = state.setdefault("s", torch.zeros_like(p))
        m.mul_(self.b1).add_((1 - self.b1) * g)
        s.mul_(self.b2).add_((1 - self.b2) * g * g)
        m_hat = m / (1 - self.b1 ** self.t)
        s_hat = s / (1 - self.b2 ** self.t)
        look_ahead = self.b1 * m_hat + (1 - self.b1) * g / (1 - self.b1 ** self.t)
        p.sub_(self.lr * look_ahead / (s_hat.sqrt() + self.eps))


for name, cls in [("Adam", Adam), ("Nadam", Nadam)]:
    model = make_mlp()
    h = train(model, cls(model.parameters(), lr=1e-3), EPOCHS, verbose=False)
    print(f"{name:6s} lr=1e-3: final train loss {h['train_loss'][-1]:.4f} | val acc {h['val_acc'][-1]:.3f}")

The two are very close on this problem, with Nadam usually a hair ahead in training loss, which matches experience: Nesterov's look-ahead is a small, cheap, consistent improvement, not a different regime.

**Exercise 3.** Show the **BatchNorm train/eval mismatch bug**. Write a `predict` function that forgets to call `model.eval()` and use the BatchNorm network `bn_model` from Section 5.4 to (a) predict a single example, (b) predict the same example inside two different batches, and (c) compare with the correct version. Explain each observation.

**Solution to Exercise 3**

In training mode BatchNorm normalizes with the statistics of the batch it is given. A batch of one has zero variance, which PyTorch refuses; and the prediction for an example then depends on which other examples happen to be in its batch, which makes the model non-deterministic and, in the extreme, wrong. `model.eval()` switches to the running statistics and removes both problems. There is a third, silent consequence: every forward pass in training mode also updates the running statistics, so a buggy prediction function slowly corrupts the model it is evaluating. We therefore run the buggy version on a copy of `bn_model`.

In [ ]:
def predict_buggy(model, x):
    with torch.no_grad():                       # gradients off, but the mode is still "train"
        return F.softmax(model(x), dim=1)


def predict_correct(model, x):
    model.eval()
    with torch.no_grad():
        return F.softmax(model(x), dim=1)


bn_buggy = copy.deepcopy(bn_model).train()      # a copy left in training mode (the copy keeps bn_model's running stats intact)
x0 = X_val[:1]
try:
    predict_buggy(bn_buggy, x0)
except ValueError as e:
    print(f"(a) single example in train mode -> ValueError: {str(e)[:80]}...")

p_in_batch_A = predict_buggy(bn_buggy, X_val[0:8])[0]              # x0 together with examples 1..7
p_in_batch_B = predict_buggy(bn_buggy, torch.cat([x0, X_val[100:107]]))[0]   # x0 with other neighbours
print(f"(b) buggy: class {p_in_batch_A.argmax().item()} (p={p_in_batch_A.max().item():.2f}) in batch A, "
      f"class {p_in_batch_B.argmax().item()} (p={p_in_batch_B.max().item():.2f}) in batch B, "
      f"max probability difference {(p_in_batch_A - p_in_batch_B).abs().max().item():.3f}")

p_alone = predict_correct(bn_model, x0)[0]
p_in_batch = predict_correct(bn_model, X_val[0:8])[0]
print(f"(c) correct: alone vs in a batch, max probability difference {(p_alone - p_in_batch).abs().max().item():.1e}; "
      f"predicted class {p_alone.argmax().item()} ({CLASSES[p_alone.argmax().item()]}), true class {CLASSES[int(y_val[0])]}")

(a) `nn.BatchNorm1d` raises "Expected more than 1 value per channel when training" for a single example, because the batch variance would be zero. (b) With eight examples the buggy prediction runs, but the probabilities for the same input change with its batch-mates, sometimes enough to change the predicted class; there is no correct answer to "what does the model predict for $x_0$". (c) In eval mode the prediction is identical whether the example is alone or in a batch. Meanwhile the running statistics of `bn_buggy` have drifted towards those of the two small batches it saw, which is the silent third consequence. The same bug affects dropout (predictions become random) and is why every evaluation function in this course starts with `model.eval()`.

**Exercise 4.** Implement **DropConnect** (Wan et al., 2013), which drops individual *weights* rather than units: during training each weight of a linear layer is zeroed with probability $p$ (and the rest scaled by $1/(1-p)$); at evaluation the full weight matrix is used. Write it as a subclass of `nn.Linear`, train a wide MLP on the 2,000-example subset with $p = 0.5$, and compare with the dropout and unregularized runs of Section 6.9.

**Solution to Exercise 4**

The mask is sampled once per forward pass on the weight matrix, so all examples in the batch share the same thinned network in this implementation (a per-example mask would need a batched matrix multiply). DropConnect is a generalization of dropout, and its effect on this problem is similar.

In [ ]:
class LinearDropConnect(nn.Linear):
    """nn.Linear whose individual weights are dropped with probability p during training (inverted scaling)."""
    def __init__(self, in_features, out_features, p=0.5):
        super().__init__(in_features, out_features)
        self.p = p

    def forward(self, x):
        if self.training and self.p > 0:
            mask = (torch.rand_like(self.weight) >= self.p).to(self.weight.dtype)
            return F.linear(x, self.weight * mask / (1 - self.p), self.bias)
        return F.linear(x, self.weight, self.bias)


torch.manual_seed(SEED)
model_dc = nn.Sequential(LinearDropConnect(784, 512, p=0.5), nn.ReLU(),
                         LinearDropConnect(512, 512, p=0.5), nn.ReLU(),
                         nn.Linear(512, 10)).to(device)
opt = torch.optim.AdamW(model_dc.parameters(), lr=1e-3, weight_decay=0.0)
h_dc = train(model_dc, opt, REG_EPOCHS, X=X_small, y=y_small, verbose=False)
print(f"{'regularizer':22s} {'train acc':>9s} {'val acc':>8s} {'best val':>9s}")
for name, h in [("DropConnect 0.5", h_dc), ("dropout 0.5", reg_rows["dropout 0.5"]), ("no regularization", reg_rows["no regularization"])]:
    print(f"{name:22s} {h['train_acc'][-1]:9.3f} {h['val_acc'][-1]:8.3f} {max(h['val_acc']):9.3f}")

DropConnect, like dropout, lowers the training accuracy and narrows the generalization gap; with weights rather than units dropped, the noise it injects is finer-grained, and the two typically end up within a fraction of a percent of each other on validation accuracy.

**Exercise 5 (challenge).** Turn the range test of Section 3.2 into a reusable **learning-rate finder** that works with any `torch.optim` optimizer: it must save the model and optimizer state before the test and restore them afterwards, drive the learning rate through `param_groups`, and suggest a learning rate automatically as the point of steepest descent of the smoothed loss curve (in log-learning-rate space). Use it to pick a learning rate for Nesterov SGD on the harness, train with a one-cycle schedule peaking at the suggested value, and compare with a default learning rate of $0.01$.

**Solution to Exercise 5**

The steepest-descent heuristic is what the `fastai` library popularized: compute the numerical derivative of the smoothed loss with respect to $\log_{10}\eta$ and take the learning rate where it is most negative. It sits safely below the minimum of the curve. Restoring the state matters: the range test ends with a diverged model, and the optimizer's momentum buffers are polluted too.

In [ ]:
class LRFinder:
    """Learning-rate range test for torch.optim optimizers, with state restoration and a suggestion."""
    def __init__(self, model, opt, loss_fn=F.cross_entropy):
        self.model, self.opt, self.loss_fn = model, opt, loss_fn
        self.lrs, self.losses = [], []

    def range_test(self, X, y, lr_min=1e-5, lr_max=1.0, steps=150, batch_size=BATCH, beta=0.98):
        model_state = copy.deepcopy(self.model.state_dict())
        opt_state = copy.deepcopy(self.opt.state_dict())
        gamma = (lr_max / lr_min) ** (1.0 / steps)
        lr, avg, best = lr_min, 0.0, float("inf")
        gen = torch.Generator().manual_seed(SEED)
        self.lrs, self.losses = [], []
        self.model.train()
        for i in range(steps):
            for group in self.opt.param_groups:
                group["lr"] = lr
            idx = torch.randint(0, len(X), (batch_size,), generator=gen).to(device)
            loss = self.loss_fn(self.model(X[idx]), y[idx])
            self.opt.zero_grad(); loss.backward(); self.opt.step()
            avg = beta * avg + (1 - beta) * loss.item()
            smoothed = avg / (1 - beta ** (i + 1))
            if not math.isfinite(smoothed) or smoothed > 4 * best:
                break
            self.lrs.append(lr); self.losses.append(smoothed)
            best = min(best, smoothed)
            lr *= gamma
        self.model.load_state_dict(model_state)          # undo the test
        self.opt.load_state_dict(opt_state)
        return self

    def suggest(self):
        """Learning rate at the steepest descent of the smoothed loss vs log10(lr)."""
        slopes = np.gradient(np.array(self.losses), np.log10(np.array(self.lrs)))
        return float(self.lrs[int(np.argmin(slopes))])

    def plot(self):
        lr_star = self.suggest()
        plt.figure(); plt.semilogx(self.lrs, self.losses)
        plt.axvline(lr_star, color="C2", ls="--", label=f"suggested lr = {lr_star:.2g}")
        plt.xlabel("learning rate (log scale)"); plt.ylabel("smoothed loss"); plt.title("LRFinder range test")
        plt.legend(); plt.tight_layout(); plt.show()


model = make_mlp()
opt = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, nesterov=True)
finder = LRFinder(model, opt).range_test(X_train, y_train, steps=RANGE_STEPS)
finder.plot()
lr_star = finder.suggest()
print(f"suggested learning rate: {lr_star:.3g}; model restored (loss on a batch back to "
      f"{F.cross_entropy(model(X_train[:BATCH]), y_train[:BATCH]).item():.3f}, i.e. near ln 10 = {math.log(10):.3f})")

results_finder = {}
for name, peak in [(f"one-cycle, peak at suggested lr {lr_star:.2g}", lr_star), ("constant default lr 0.01", None)]:
    model = make_mlp()
    opt = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, nesterov=True)
    sched = OneCycleLR(opt, max_lr=peak, total_steps=T_TOTAL) if peak is not None else None
    results_finder[name] = train(model, opt, EPOCHS, scheduler=sched, verbose=False)
    print(f"{name:42s}: final train loss {results_finder[name]['train_loss'][-1]:.4f} | val acc {results_finder[name]['val_acc'][-1]:.3f}")

The finder proposes a learning rate on the steep part of the curve, and the one-cycle run that peaks there typically beats the default learning rate at the same budget, both in training loss and in validation accuracy. The whole procedure costs one short extra run, which is why "range test, then one-cycle or cosine" is such a common recipe for a new problem.

## Summary

- Mini-batch gradients are unbiased but noisy estimates of the full gradient; the noise scales with $\eta / b$, and controlling it (smaller learning rate late in training, larger batches with proportionally larger learning rates) is the recurring theme of optimization.
- Momentum accumulates an exponentially weighted sum of past gradients, which speeds up travel along consistent directions and damps oscillation across steep ones; Nesterov's look-ahead is a cheap refinement, implemented in PyTorch as a step with $g_t + \beta v_t$.
- Adaptive methods (AdaGrad, RMSProp, Adam) divide each coordinate by a running RMS of its gradient, giving per-parameter learning rates and scale invariance. Adam's bias correction divides the moving averages by $1 - \beta^t$, which we derived from the geometric series; without it the early steps would be several times too large.
- Weight decay and L2 regularization coincide for SGD but not for Adam; AdamW's decoupled decay applies the same shrinkage to every weight regardless of its gradient scale, and is the standard for Transformers.
- Learning-rate schedules (step, cosine, warmup + cosine, one-cycle) matter as much as the optimizer; always decay towards the end, and use a range test to choose the initial value.
- Variance propagation gives the initialization rules $\sigma_W^2 = 2/(n_{\text{in}} + n_{\text{out}})$ (Xavier, for tanh) and $2/n_{\text{in}}$ (He, for ReLU); with the wrong scale, activations and gradients vanish or explode exponentially with depth, as we saw in 10- and 20-layer networks.
- BatchNorm normalizes each feature over the batch with learnable scale and shift, uses running statistics in eval mode, makes deep networks trainable at large learning rates, and degrades with tiny batches; LayerNorm normalizes each example over its features and is the choice for sequence models.
- Regularizers (weight decay, dropout, early stopping, augmentation, label smoothing) trade a little training accuracy for a smaller generalization gap; augmentation is usually the strongest on image data, and none of them helps an underfitting model.
- Loss curves have recognizable shapes for too-low and too-high learning rates, bad initialization, overfitting and underfitting; the overfit-one-batch test catches pipeline bugs in seconds; `nan` losses almost always come from the learning rate, a log or division of zero, or overflow.
- Random search with log-uniform sampling beats grid search for the same budget; decide on the validation set, and touch the test set exactly once.

## Further reading

- Goodfellow, Bengio and Courville (2016), *Deep Learning*, Chapter 7 (Regularization) and Chapter 8 (Optimization for Training Deep Models). https://www.deeplearningbook.org
- Zhang, Lipton, Li and Smola, *Dive into Deep Learning*, Chapter 12 (Optimization Algorithms). https://d2l.ai
- Ruder (2016), *An overview of gradient descent optimization algorithms*, arXiv:1609.04747.
- Sutskever, Martens, Dahl and Hinton (2013), *On the importance of initialization and momentum in deep learning*, ICML.
- Duchi, Hazan and Singer (2011), *Adaptive Subgradient Methods for Online Learning and Stochastic Optimization*, JMLR.
- Kingma and Ba (2015), *Adam: A Method for Stochastic Optimization*, ICLR.
- Loshchilov and Hutter (2019), *Decoupled Weight Decay Regularization*, ICLR; and (2017), *SGDR: Stochastic Gradient Descent with Warm Restarts*, ICLR.
- Smith (2017), *Cyclical Learning Rates for Training Neural Networks*, WACV; Smith and Topin (2019), *Super-convergence: very fast training of neural networks using large learning rates*.
- Glorot and Bengio (2010), *Understanding the difficulty of training deep feedforward neural networks*, AISTATS; He, Zhang, Ren and Sun (2015), *Delving Deep into Rectifiers*, ICCV.
- Ioffe and Szegedy (2015), *Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift*, ICML; Santurkar, Tsipras, Ilyas and Madry (2018), *How Does Batch Normalization Help Optimization?*, NeurIPS; Ba, Kiros and Hinton (2016), *Layer Normalization*.
- Srivastava, Hinton, Krizhevsky, Sutskever and Salakhutdinov (2014), *Dropout: A Simple Way to Prevent Neural Networks from Overfitting*, JMLR.
- Bergstra and Bengio (2012), *Random Search for Hyper-Parameter Optimization*, JMLR.
- PyTorch documentation: `torch.optim` and learning-rate schedulers, https://docs.pytorch.org/docs/stable/optim.html; `torch.nn.init`, https://docs.pytorch.org/docs/stable/nn.init.html